## 0. Cài thư viện

In [ ]:
%pip install -q \
    pandas==2.2.2 numpy==1.26.4 tqdm==4.66.4 pydantic==2.7.4 python-dotenv==1.0.1 \
    neo4j==5.19.0 qdrant-client==1.9.1 sentence-transformers==3.0.1 torch==2.3.1 transformers==4.44.2 \
    langchain==0.2.6 langchain-openai==0.1.13 langchain-anthropic==0.1.20 \
    python-igraph==0.11.6 FlagEmbedding==1.2.11
print("✅ Dependencies installed")

## 1. Imports, config và đường dẫn

In [ ]:
from __future__ import annotations
import os, re, json, ast, math, hashlib, unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Literal
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError

load_dotenv(dotenv_path=Path.cwd() / ".env")

REPO_ROOT = Path.cwd()
DATA_ROOT = Path(os.getenv("DATA_ROOT", str(REPO_ROOT / "Utils")))

BEFOOD_RESTAURANTS_PATH = Path(os.getenv("BEFOOD_RESTAURANTS_PATH", str(DATA_ROOT / "befood_bachkhoa_restaurants.csv")))
BEFOOD_MENU_PATH = Path(os.getenv("BEFOOD_MENU_PATH", str(DATA_ROOT / "befood_bachkhoa_menu_items.csv")))
FOODY_PATH = Path(os.getenv("FOODY_PATH", str(DATA_ROOT / "foody_hust_places_from_store_csv.csv")))

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", os.getenv("NEO4J_USERNAME", "neo4j"))
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

QDRANT_HOST = os.getenv("QDRANT_HOST", "localhost")
QDRANT_PORT = int(os.getenv("QDRANT_PORT", 6333))
COLL_TEXT_UNIT = os.getenv("COLL_TEXT_UNIT", "graphrag_text_units_vietnamese")
COLL_RESTAURANT = os.getenv("COLL_RESTAURANT", "graphrag_restaurants_vietnamese")

EMBED_MODEL = os.getenv("EMBED_MODEL", "bkai-foundation-models/vietnamese-bi-encoder")
EMBED_PREFIX_QUERY = os.getenv("EMBED_PREFIX_QUERY", "")
EMBED_PREFIX_PASSAGE = os.getenv("EMBED_PREFIX_PASSAGE", "")
ASPECT_SENTIMENT_MODEL = os.getenv("ASPECT_SENTIMENT_MODEL", "wonrax/phobert-base-vietnamese-sentiment")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "openai").lower()
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-20250514")

COMMUNITY_LEVEL = int(os.getenv("COMMUNITY_LEVEL", "0"))
RRF_K = int(os.getenv("RRF_K", "60"))
SIMILARITY_TOP_K = int(os.getenv("SIMILARITY_TOP_K", "8"))
SIMILARITY_MIN_SCORE = float(os.getenv("SIMILARITY_MIN_SCORE", "0.62"))

def env_float(name: str, default: Optional[float] = None) -> Optional[float]:
    raw = os.getenv(name)
    if raw is None or str(raw).strip() == "":
        return default
    try:
        return float(raw)
    except ValueError:
        return default

# Optional per-session user location. Set these from an app request, notebook cell, or .env.
USER_LAT = env_float("USER_LAT")
USER_LNG = env_float("USER_LNG")
MAX_DISTANCE_KM = env_float("MAX_DISTANCE_KM")
DISTANCE_WEIGHT = float(os.getenv("DISTANCE_WEIGHT", "0.20"))
DISTANCE_DECAY_KM = float(os.getenv("DISTANCE_DECAY_KM", "3.0"))
RECREATE_QDRANT = os.getenv("RECREATE_QDRANT", "true").lower() in {"1", "true", "yes", "y"}
RUN_COMMUNITY_REPORTS = os.getenv("RUN_COMMUNITY_REPORTS", "true").lower() in {"1", "true", "yes", "y"}

print("Config loaded")
print("Repo root:", REPO_ROOT)
print("Data files:", BEFOOD_RESTAURANTS_PATH, BEFOOD_MENU_PATH, FOODY_PATH, sep="\n  ")
print("Neo4j:", NEO4J_URI, "user=", NEO4J_USER)
print("Qdrant:", f"{QDRANT_HOST}:{QDRANT_PORT}")
print("Embedding:", EMBED_MODEL)
print("Aspect sentiment:", ASPECT_SENTIMENT_MODEL)
print("User location:", USER_LAT, USER_LNG, "max_distance_km=", MAX_DISTANCE_KM)
print("Recreate Qdrant:", RECREATE_QDRANT, "run community reports:", RUN_COMMUNITY_REPORTS)

missing = [str(x) for x in [BEFOOD_RESTAURANTS_PATH, BEFOOD_MENU_PATH, FOODY_PATH] if not x.exists()]
if missing:
    raise FileNotFoundError("Missing required data file(s):\n" + "\n".join(missing))


In [ ]:
# Define get_llm function manually
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

def get_llm():
    if LLM_PROVIDER == "anthropic":
        if not ANTHROPIC_API_KEY:
            raise RuntimeError("LLM_PROVIDER=anthropic but ANTHROPIC_API_KEY is missing.")
        return ChatAnthropic(model=ANTHROPIC_MODEL, api_key=ANTHROPIC_API_KEY, temperature=0)
    if LLM_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("LLM_PROVIDER=openai but OPENAI_API_KEY is missing.")
        return ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0)
    raise ValueError(f"Unsupported LLM_PROVIDER={LLM_PROVIDER}. Use 'openai' or 'anthropic'.")

# Test get_llm function
try:
    llm = get_llm()
    print("✅ LLM initialized successfully")
    print(f"LLM type: {type(llm)}")
except Exception as e:
    print(f"❌ Error initializing LLM: {e}")

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=OPENAI_BASE_URL
    
)

try:
    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": "Xin chào"
            }
        ]
    )

    print("✅ API hoạt động")
    print(response.choices[0].message.content)

except Exception as e:
    print("❌ API lỗi")
    print(type(e).__name__)
    print(e)

## 2. Inspect dữ liệu thật

In [ ]:
raw_befood = pd.read_csv(BEFOOD_RESTAURANTS_PATH)
raw_menu = pd.read_csv(BEFOOD_MENU_PATH)
raw_foody = pd.read_csv(FOODY_PATH)

display(raw_befood.head(2))
display(raw_menu.head(2))
display(raw_foody.head(2))

print("Shapes:")
print("  raw_befood:", raw_befood.shape)
print("  raw_menu  :", raw_menu.shape)
print("  raw_foody :", raw_foody.shape)


In [ ]:
def _is_nan(x: Any) -> bool:
    return pd.isna(x)

def normalize_text(x: Any) -> str:
    if x is None or _is_nan(x):
        return ""
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKC", x)
    return re.sub(r"\s+", " ", x)

def slugify_vn(x: Any) -> str:
    x = normalize_text(x)
    x = ''.join(c for c in unicodedata.normalize('NFD', x) if unicodedata.category(c) != 'Mn')
    x = x.replace("đ", "d")
    x = re.sub(r"[^a-z0-9]+", "-", x).strip("-")
    return x

def to_float(x: Any) -> Optional[float]:
    if x is None or _is_nan(x) or str(x).strip() == "":
        return None
    try:
        return float(x)
    except:
        s = str(x).replace(".", "").replace(",", ".")
        s = re.sub(r"[^0-9.\-]", "", s)
        try:
            return float(s)
        except:
            return None

def to_int(x: Any) -> Optional[int]:
    v = to_float(x)
    return None if v is None else int(v)

def parse_jsonish(x: Any) -> Any:
    if x is None or _is_nan(x):
        return None
    if isinstance(x, (list, dict)):
        return x
    s = str(x).strip()
    if not s:
        return None
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(s)
        except:
            pass
    return s

def split_semi(x: Any) -> List[str]:
    if x is None or _is_nan(x):
        return []
    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]
    parsed = parse_jsonish(x)
    if isinstance(parsed, list):
        return [str(i).strip() for i in parsed if str(i).strip()]
    s = str(x)
    parts = re.split(r"[;|,/]", s)
    return [p.strip() for p in parts if p and p.strip()]

def parse_price_band(text: Any) -> Optional[str]:
    if text is None or _is_nan(text):
        return None
    s = normalize_text(text)
    if not s:
        return None
    nums = re.findall(r"\d[\d\.]*", s)
    vals = []
    for n in nums:
        try:
            vals.append(int(float(n.replace(".", ""))))
        except:
            pass
    if not vals:
        return None
    hi = max(vals)
    if hi <= 50000:
        return "budget"
    if hi <= 120000:
        return "mid"
    return "premium"

def set_user_location(lat: Optional[float], lng: Optional[float], max_distance_km: Optional[float] = None):
    """Set current user location for distance-aware retrieval in this notebook session."""
    global USER_LAT, USER_LNG, MAX_DISTANCE_KM
    USER_LAT = to_float(lat)
    USER_LNG = to_float(lng)
    if max_distance_km is not None:
        MAX_DISTANCE_KM = to_float(max_distance_km)


def get_user_location(user_lat: Optional[float] = None, user_lng: Optional[float] = None) -> Tuple[Optional[float], Optional[float]]:
    lat = to_float(user_lat) if user_lat is not None else USER_LAT
    lng = to_float(user_lng) if user_lng is not None else USER_LNG
    return lat, lng


def haversine_km(lat1: Any, lng1: Any, lat2: Any, lng2: Any) -> Optional[float]:
    lat1, lng1, lat2, lng2 = map(to_float, [lat1, lng1, lat2, lng2])
    if None in (lat1, lng1, lat2, lng2):
        return None
    r = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lng2 - lng1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return round(2 * r * math.asin(math.sqrt(a)), 3)


def distance_score(distance_km: Any, decay_km: float = DISTANCE_DECAY_KM) -> float:
    d = to_float(distance_km)
    if d is None:
        return 0.0
    return 1.0 / (1.0 + max(d, 0.0) / max(decay_km, 1e-9))


def add_user_distance_to_record(rec: dict, user_lat: Optional[float] = None, user_lng: Optional[float] = None) -> dict:
    lat, lng = get_user_location(user_lat, user_lng)
    if lat is None or lng is None:
        rec.setdefault("distance_km", None)
        rec.setdefault("distance_score", 0.0)
        return rec
    dist = haversine_km(lat, lng, rec.get("lat"), rec.get("lng"))
    rec["distance_km"] = dist
    rec["distance_score"] = distance_score(dist)
    return rec



## 3. Canonicalize schema và hợp nhất nguồn dữ liệu

In [ ]:
HANOI_DISTRICTS = [
    "Ba Dinh", "Hoan Kiem", "Tay Ho", "Long Bien", "Cau Giay", "Dong Da", "Hai Ba Trung",
    "Hoang Mai", "Thanh Xuan", "Nam Tu Liem", "Bac Tu Liem", "Ha Dong", "Son Tay",
    "Ba Vi", "Chuong My", "Dan Phuong", "Dong Anh", "Gia Lam", "Hoai Duc", "Me Linh",
    "My Duc", "Phu Xuyen", "Phuc Tho", "Quoc Oai", "Soc Son", "Thach That", "Thanh Oai",
    "Thanh Tri", "Thuong Tin", "Ung Hoa",
]
URBAN_DISTRICTS = {"Ba Dinh", "Hoan Kiem", "Tay Ho", "Long Bien", "Cau Giay", "Dong Da", "Hai Ba Trung", "Hoang Mai", "Thanh Xuan", "Nam Tu Liem", "Bac Tu Liem", "Ha Dong"}
DISTRICT_ALIASES = {slugify_vn(x): (f"Quan {x}" if x in URBAN_DISTRICTS else x) for x in HANOI_DISTRICTS}

def infer_district(address: Any) -> Optional[str]:
    s = slugify_vn(address)
    for key, label in DISTRICT_ALIASES.items():
        if key and key in s:
            return label
    return None

def price_band_from_bounds(price_min: Any, price_max: Any) -> Optional[str]:
    vals = [to_float(price_min), to_float(price_max)]
    vals = [v for v in vals if v is not None]
    if not vals:
        return None
    hi = max(vals)
    if hi <= 50000:
        return "budget"
    if hi <= 120000:
        return "mid"
    return "premium"

def price_band_from_menu_prices(prices: Any) -> Optional[str]:
    vals = [to_float(x) for x in list(prices) if to_float(x) is not None and to_float(x) > 0]
    if not vals:
        return None
    median = float(np.median(vals))
    budget_ratio = sum(v <= 50000 for v in vals) / len(vals)
    premium_ratio = sum(v > 120000 for v in vals) / len(vals)
    # Median and item distribution are more robust than max price because menus often contain combo/outlier items.
    if median <= 50000 or budget_ratio >= 0.60:
        return "budget"
    if median <= 120000 and premium_ratio < 0.35:
        return "mid"
    return "premium"


def normalize_dish_family(name: Any) -> Optional[str]:
    """Normalize exact menu item names into broader dish families.

    Examples: "C?m g? s?t chua ng?t" -> "c?m g?", "Ph? b? t?i" -> "ph?",
    "B?nh cu?n ch?" -> "b?nh cu?n". This is intentionally broader than exact MenuItem.
    """
    raw = normalize_text(name)
    s = slugify_vn(raw)
    if not s:
        return None
    rules = [
        (r"com-tam", "c?m t?m"),
        (r"com-ga", "c?m g?"),
        (r"com-rang", "c?m rang"),
        (r"com", "c?m"),
        (r"bun-cha", "b?n ch?"),
        (r"bun-bo", "b?n b?"),
        (r"bun-ca", "b?n c?"),
        (r"bun-rieu", "b?n ri?u"),
        (r"bun", "b?n"),
        (r"pho", "ph?"),
        (r"banh-cuon", "b?nh cu?n"),
        (r"banh-mi", "b?nh m?"),
        (r"ga-ran", "g? r?n"),
        (r"ga", "g?"),
        (r"mi-cay", "m? cay"),
        (r"mi", "m?"),
        (r"mien", "mi?n"),
        (r"chao", "ch?o"),
        (r"xoi", "x?i"),
        (r"lau", "l?u"),
        (r"nuong", "n??ng"),
        (r"tra-sua", "tr? s?a"),
        (r"cafe|ca-phe", "c? ph?"),
        (r"nuoc-ep", "n??c ?p"),
    ]
    for pattern, family in rules:
        if re.search(pattern, s):
            return family
    # Fallback: remove common modifiers and keep a compact 1-2 token family.
    cleaned = re.sub(r"(combo|set|size|phan|them|coca|cola|pepsi|sprite|dasani|lon|chai|dac-biet|full|mix)", " ", s)
    tokens = [t for t in cleaned.split("-") if t and not t.isdigit()]
    if not tokens:
        return None
    return " ".join(tokens[:2])

def canonicalize_befood_restaurants(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["store_id"] = df["restaurant_id"].astype("Int64").astype(str)
    out["store_key"] = out["store_id"]
    out["store_name"] = df["restaurant_name"].fillna("").astype(str)
    out["query_name"] = out["store_name"]
    out["address"] = df.get("address", pd.Series([None] * len(df)))
    out["query_address"] = out["address"]
    out["district"] = out["address"].apply(infer_district)
    out["city"] = "Ha Noi"
    out["lat"] = df.get("latitude", pd.Series([None] * len(df))).apply(to_float)
    out["lng"] = df.get("longitude", pd.Series([None] * len(df))).apply(to_float)
    out["gmaps_rating"] = df.get("rating", pd.Series([None] * len(df))).apply(to_float)
    out["gmaps_review_count"] = df.get("review_count", pd.Series([None] * len(df))).apply(to_int)
    out["price_min"] = df.get("price_min", pd.Series([None] * len(df))).apply(to_float)
    out["price_max"] = df.get("price_max", pd.Series([None] * len(df))).apply(to_float)
    out["price_band"] = [price_band_from_bounds(a, b) for a, b in zip(out["price_min"], out["price_max"])]
    out["categories"] = df.get("categories_text", pd.Series([None] * len(df))).apply(split_semi)
    out["matched_terms"] = df.get("matched_terms_text", pd.Series([None] * len(df))).apply(split_semi)
    out["opening_hours_raw"] = df.get("opening_hours")
    out["delivery_time"] = df.get("delivery_time", pd.Series([None] * len(df))).apply(to_float)
    out["image_url"] = df.get("image_url")
    out["menu_count"] = df.get("menu_count", pd.Series([None] * len(df))).apply(to_int)
    out["comment_count"] = df.get("comment_count", pd.Series([None] * len(df))).apply(to_int)
    out["source"] = df.get("source", pd.Series(["befood"] * len(df))).fillna("befood")
    out["atmosphere"] = [[] for _ in range(len(out))]
    out["crowd"] = [[] for _ in range(len(out))]
    out["name_norm"] = out["store_name"].apply(slugify_vn)
    out["addr_norm"] = out["address"].apply(slugify_vn)
    return out

def canonicalize_feedback_from_befood(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in df.iterrows():
        store_id = str(int(r["restaurant_id"])) if pd.notna(r.get("restaurant_id")) else ""
        store_name = str(r.get("restaurant_name") or "")
        comments = parse_jsonish(r.get("comments_list"))
        if isinstance(comments, str):
            comments = [comments]
        if not isinstance(comments, list):
            comments = []
        for i, comment in enumerate(comments):
            feedback = str(comment or "").strip()
            if not feedback:
                continue
            rows.append({
                "store_id": store_id,
                "store_key": store_id,
                "store_name": store_name,
                "rated_at": None,
                "rating": to_float(r.get("rating")) or 3.0,
                "feedback": feedback,
                "source": "befood_comment",
                "review_id": hashlib.md5(f"{store_id}|{i}|{feedback}".encode("utf-8")).hexdigest()[:16],
                "name_norm": slugify_vn(store_name),
            })
    return pd.DataFrame(rows, columns=["store_id", "store_key", "store_name", "rated_at", "rating", "feedback", "source", "review_id", "name_norm"])

def canonicalize_menu_items(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["store_id"] = df["restaurant_id"].astype("Int64").astype(str)
    out["store_key"] = out["store_id"]
    out["store_name"] = df["restaurant_name"].fillna("").astype(str)
    out["category_id"] = df.get("category_id", pd.Series([None] * len(df))).astype("Int64").astype(str)
    out["category_name"] = df.get("category_name", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()
    out["menu_item_id"] = df["restaurant_item_id"].astype("Int64").astype(str)
    out["item_name"] = df["item_name"].fillna("").astype(str).str.strip()
    out["item_details"] = df.get("item_details", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()
    out["price"] = df.get("price", pd.Series([None] * len(df))).apply(to_float)
    out["old_price"] = df.get("old_price", pd.Series([None] * len(df))).apply(to_float)
    out["order_count"] = df.get("order_count", pd.Series([0] * len(df))).apply(to_int).fillna(0).astype(int)
    out["like_count"] = df.get("like_count", pd.Series([0] * len(df))).apply(to_int).fillna(0).astype(int)
    out["dislike_count"] = df.get("dislike_count", pd.Series([0] * len(df))).apply(to_int).fillna(0).astype(int)
    out["category_position"] = df.get("category_position", pd.Series([None] * len(df))).apply(to_int)
    out["item_position"] = df.get("item_position", pd.Series([None] * len(df))).apply(to_int)
    out["item_image"] = df.get("item_image")
    out["item_norm"] = out["item_name"].apply(slugify_vn)
    return out[out["item_name"].ne("")].copy()

def canonicalize_foody(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["input_store_id"] = df["input_store_id"].astype("Int64").astype(str)
    out["input_store_name"] = df["input_store_name"]
    out["crawl_status"] = df.get("crawl_status")
    out["foody_name"] = df.get("name")
    out["foody_address"] = df.get("address")
    out["district"] = df.get("district")
    out["area"] = df.get("area")
    out["city"] = df.get("city")
    out["foody_lat"] = df.get("lat").apply(to_float)
    out["foody_lng"] = df.get("lng").apply(to_float)
    out["foody_rating"] = df.get("avg_rating").apply(to_float)
    out["foody_review_count"] = df.get("total_review").apply(to_int)
    out["price_min"] = df.get("price_min").apply(to_float)
    out["price_max"] = df.get("price_max").apply(to_float)
    out["foody_price_band"] = [price_band_from_bounds(a, b) for a, b in zip(out["price_min"], out["price_max"])]
    out["categories"] = df.get("categories", pd.Series([None]*len(df))).apply(split_semi)
    out["cuisines"] = df.get("cuisines", pd.Series([None]*len(df))).apply(split_semi)
    out["audiences"] = df.get("audiences", pd.Series([None]*len(df))).apply(split_semi)
    out["opening_hours_foody"] = df.get("opening_hours")
    out["rating_quality"] = df.get("rating_quality").apply(to_float)
    out["rating_position"] = df.get("rating_position").apply(to_float)
    out["rating_service"] = df.get("rating_service").apply(to_float)
    out["rating_price"] = df.get("rating_price").apply(to_float)
    out["rating_space"] = df.get("rating_space").apply(to_float)
    out["store_key"] = out["input_store_id"].astype(str)
    out["name_norm"] = out["input_store_name"].apply(slugify_vn)
    return out

restaurants_base = canonicalize_befood_restaurants(raw_befood)
feedback = canonicalize_feedback_from_befood(raw_befood)
menu_items = canonicalize_menu_items(raw_menu)
foody = canonicalize_foody(raw_foody)
gmaps = restaurants_base
print(f"Canonicalized: restaurants={len(restaurants_base)}, menu_items={len(menu_items)}, comments={len(feedback)}, foody_rows={len(foody)}")


In [ ]:
restaurants = restaurants_base.merge(
    foody.drop_duplicates("store_key"),
    on="store_key",
    how="left",
    suffixes=("", "_foody")
)

restaurants["name"] = restaurants["store_name"].fillna(restaurants["foody_name"]).fillna(restaurants["query_name"])
restaurants["address_final"] = restaurants["address"].fillna(restaurants["foody_address"]).fillna(restaurants["query_address"])
restaurants["district_final"] = restaurants["district_foody"].fillna(restaurants["district"])
restaurants["city_final"] = restaurants["city_foody"].fillna(restaurants["city"]).fillna("Ha Noi")
restaurants["lat_final"] = restaurants["lat"].fillna(restaurants["foody_lat"])
restaurants["lng_final"] = restaurants["lng"].fillna(restaurants["foody_lng"])
restaurants["source_price_band_final"] = restaurants["price_band"].fillna(restaurants["foody_price_band"])

def merge_list_cols(*cols):
    out, seen = [], set()
    for col in cols:
        vals = col if isinstance(col, list) else []
        for x in vals:
            x = str(x).strip()
            key = x.lower()
            if x and key not in seen:
                out.append(x)
                seen.add(key)
    return out

menu_categories_by_store = menu_items.groupby("store_key")["category_name"].apply(lambda s: sorted({x for x in s if x})).to_dict()
menu_top_items_by_store = (
    menu_items.sort_values(["store_key", "order_count", "like_count"], ascending=[True, False, False])
    .groupby("store_key")
    .head(12)
    .groupby("store_key")["item_name"]
    .apply(list)
    .to_dict()
)
menu_price_stats = menu_items.groupby("store_key").agg(
    menu_item_count=("menu_item_id", "count"),
    menu_price_min=("price", "min"),
    menu_price_max=("price", "max"),
    menu_price_median=("price", "median"),
    menu_budget_item_ratio=("price", lambda s: float((s <= 50000).mean()) if len(s) else 0.0),
    menu_price_band=("price", price_band_from_menu_prices),
).reset_index()

restaurants["categories_final"] = [
    merge_list_cols(a, b, menu_categories_by_store.get(k, []), terms)
    for a, b, k, terms in zip(restaurants["categories"], restaurants["categories_foody"], restaurants["store_key"], restaurants["matched_terms"])
]
restaurants["cuisines_final"] = restaurants["cuisines"].apply(lambda x: x if isinstance(x, list) else [])
restaurants["audiences_final"] = restaurants["audiences"].apply(lambda x: x if isinstance(x, list) else [])

summary = pd.DataFrame({
    "store_key": restaurants["store_key"],
    "name": restaurants["name"],
    "address": restaurants["address_final"],
    "district": restaurants["district_final"],
    "city": restaurants["city_final"],
    "gmaps_rating": restaurants["gmaps_rating"],
    "foody_rating": restaurants["foody_rating"],
    "review_count": restaurants["gmaps_review_count"].fillna(restaurants["foody_review_count"]),
    "source_price_band": restaurants["source_price_band_final"],
    "price_min": restaurants["price_min"],
    "price_max": restaurants["price_max"],
    "categories": restaurants["categories_final"],
    "cuisines": restaurants["cuisines_final"],
    "atmosphere": restaurants["atmosphere"],
    "audiences": restaurants["audiences_final"],
    "opening_hours": restaurants["opening_hours_raw"].fillna(restaurants["opening_hours_foody"]),
    "delivery_time": restaurants["delivery_time"],
    "image_url": restaurants["image_url"],
    "top_menu_items": restaurants["store_key"].map(menu_top_items_by_store).apply(lambda x: x if isinstance(x, list) else []),
    "lat": restaurants["lat_final"],
    "lng": restaurants["lng_final"],
}).merge(menu_price_stats, on="store_key", how="left")

if USER_LAT is not None and USER_LNG is not None:
    summary["distance_km"] = [haversine_km(USER_LAT, USER_LNG, lat, lng) for lat, lng in zip(summary["lat"], summary["lng"])]
else:
    summary["distance_km"] = None
summary["distance_score"] = summary["distance_km"].apply(distance_score)

summary["menu_item_count"] = summary["menu_item_count"].fillna(0).astype(int)
summary["price_band"] = summary["menu_price_band"].fillna(summary["source_price_band"])
summary["rating"] = summary["gmaps_rating"].fillna(summary["foody_rating"])

display(summary.head(10))
print("Unified restaurant table ready:", summary.shape)
print("Feedback comments ready:", feedback.shape)
print("Menu items ready:", menu_items.shape)


## 3.1. Menu-derived entity extraction

In [ ]:
from langchain.prompts import ChatPromptTemplate

In [ ]:
# Deprecated: dish/entity extraction is now built from befood_bachkhoa_menu_items.csv in the menu-derived entity cell below.

## 4. PhoBERT-based aspect sentiment và TextUnit evidence


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ASPECTS: Dict[str, str] = {
    "food_quality": "chất lượng món ăn, độ ngon, hương vị, độ tươi",
    "service": "thái độ và chất lượng phục vụ của nhân viên",
    "cleanliness": "vệ sinh, sạch sẽ, mùi, an toàn thực phẩm",
    "packaging": "đóng gói, giao hàng không đổ vỡ, đầy đủ món",
    "price": "giá cả, độ đáng tiền, phù hợp sinh viên",
    "space": "không gian quán, độ rộng, yên tĩnh, thoải mái",
    "speed": "tốc độ phục vụ, thời gian chờ món hoặc giao hàng",
}

LABEL_TO_SCORE = {
    "negative": -1.0, "neg": -1.0, "0": -1.0,
    "neutral": 0.0, "neu": 0.0, "1": 0.0,
    "positive": 1.0, "pos": 1.0, "2": 1.0,
}

def normalize_review_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = unicodedata.normalize("NFKC", s.lower())
    return re.sub(r"\s+", " ", s).strip()

class PhoBERTAspectSentiment:
    """Aspect sentiment with batched multi-aspect inference.

    FIX (Vấn đề 6): Thay vì gọi model 7 lần per review (7 aspects × N reviews = 7N calls),
    ta build 7×N texts trước, rồi chạy batched inference 1 lần dùng tokenizer padding.
    Giảm 5–10× thời gian indexing so với sequential forward pass.
    """
    def __init__(self, model_name: str, device: Optional[str] = None, batch_size: int = 64):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()
        self.id2label = {int(k): str(v).lower() for k, v in self.model.config.id2label.items()}
        if not self.id2label:
            raise RuntimeError(f"Model {model_name} does not expose id2label; cannot map sentiment labels safely.")
        self.aspect_names = list(ASPECTS.keys())
        self.aspect_descs = list(ASPECTS.values())

    @torch.inference_mode()
    def _batch_infer(self, texts: List[str]) -> List[float]:
        """Run batched inference, return expected sentiment score per text."""
        scores = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            enc = self.tokenizer(
                batch,
                truncation=True,
                max_length=256,
                padding=True,
                return_tensors="pt",
            ).to(self.device)
            logits = self.model(**enc).logits  # (B, n_labels)
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
            for row in probs:
                expected = 0.0
                mass = 0.0
                for j, p in enumerate(row):
                    label = self.id2label.get(j, str(j)).lower()
                    score = LABEL_TO_SCORE.get(label)
                    if score is None:
                        raise RuntimeError(f"Unsupported label {label} from {self.model_name}. Update LABEL_TO_SCORE.")
                    expected += float(p) * score
                    mass += float(p)
                scores.append(round(expected / max(mass, 1e-9), 4))
        return scores

    def score_reviews_batch(self, reviews: List[str]) -> List[Dict[str, float]]:
        """Score all aspects for all reviews in one batched forward pass sequence.

        Build (n_aspects × n_reviews) texts, infer in batches, reshape.
        This is O(n_aspects * ceil(N/batch)) batches instead of O(n_aspects * N) sequential calls.
        """
        aspect_texts = []
        for asp_desc in self.aspect_descs:
            for review in reviews:
                aspect_texts.append(f"Khía cạnh: {asp_desc}. Nhận xét: {review}")

        all_scores = self._batch_infer(aspect_texts)

        n = len(reviews)
        results = []
        for review_idx in range(n):
            result = {}
            for asp_idx, asp_name in enumerate(self.aspect_names):
                result[asp_name] = all_scores[asp_idx * n + review_idx]
            results.append(result)
        return results

    # backward compat: single review
    def score_review(self, review: str) -> Dict[str, float]:
        return self.score_reviews_batch([review])[0]

aspect_model = PhoBERTAspectSentiment(ASPECT_SENTIMENT_MODEL)

def classify_sentiment_from_aspects(aspect_scores: Dict[str, float]) -> str:
    avg = float(np.mean(list(aspect_scores.values()))) if aspect_scores else 0.0
    if avg >= 0.20:
        return "positive"
    if avg <= -0.20:
        return "negative"
    return "neutral"

feedback_proc = feedback.copy()
feedback_proc["feedback_norm"] = feedback_proc["feedback"].apply(normalize_review_text)

# Batched scoring: 1 call per aspect × ceil(N/batch) instead of 7 calls per review
print("Running batched PhoBERT aspect inference...")
reviews_list = feedback_proc["feedback_norm"].tolist()
aspect_scores_list = aspect_model.score_reviews_batch(reviews_list)
feedback_proc["aspect_scores"] = aspect_scores_list
feedback_proc["sentiment"] = feedback_proc["aspect_scores"].apply(classify_sentiment_from_aspects)
print(f"✅ PhoBERT batched inference done: {len(feedback_proc)} reviews")
feedback_proc.head(3)

In [ ]:
# Menu-derived dish family extraction.
# MenuItem remains exact; DishFamily is the broad entity used for retrieval constraints.

def build_menu_dish_families(menu_df: pd.DataFrame) -> pd.DataFrame:
    if menu_df.empty:
        return pd.DataFrame(columns=["store_key", "dish_family", "total_menu_items", "avg_price", "order_count", "like_count", "menu_item_ids", "example_items"])
    df = menu_df.copy()
    df["dish_family"] = df["item_name"].apply(normalize_dish_family)
    df = df[df["dish_family"].notna() & df["dish_family"].ne("")].copy()
    return df.groupby(["store_key", "dish_family"]).agg(
        total_menu_items=("menu_item_id", "count"),
        avg_price=("price", "mean"),
        order_count=("order_count", "sum"),
        like_count=("like_count", "sum"),
        menu_item_ids=("menu_item_id", lambda s: [str(x) for x in s]),
        example_items=("item_name", lambda s: list(dict.fromkeys([str(x) for x in s]))[:5]),
    ).reset_index()


def upsert_dish_families(client, family_df: pd.DataFrame):
    """Write broad DishFamily nodes and (Restaurant)-[:SERVES_FAMILY]->(DishFamily) edges."""
    if family_df.empty:
        print("No dish families to upsert")
        return
    rows = family_df.rename(columns={"dish_family": "name"}).to_dict("records")
    client.run("""
    UNWIND $rows AS row
    MATCH (r:Restaurant {store_key: row.store_key})
    MERGE (d:DishFamily {name: row.name})
    MERGE (r)-[s:SERVES_FAMILY]->(d)
    SET s.menu_item_count = row.total_menu_items,
        s.like_count = row.like_count,
        s.order_count = row.order_count,
        s.avg_price = row.avg_price,
        s.menu_item_ids = row.menu_item_ids,
        s.example_items = row.example_items,
        s.updated_at = datetime()
    """, {"rows": rows})
    print(f"Upserted {len(family_df)} dish family edges")


def upsert_menu_items_graph(client, menu_df: pd.DataFrame):
    """Write concrete MenuItem nodes linked to Restaurant and MenuCategory."""
    if menu_df.empty:
        print("No menu items to upsert")
        return
    rows = menu_df.to_dict("records")
    client.run("""
    UNWIND $rows AS row
    MATCH (r:Restaurant {store_key: row.store_key})
    MERGE (mi:MenuItem {menu_item_id: row.menu_item_id})
    SET mi.name = row.item_name,
        mi.details = row.item_details,
        mi.price = row.price,
        mi.old_price = row.old_price,
        mi.order_count = row.order_count,
        mi.like_count = row.like_count,
        mi.dislike_count = row.dislike_count,
        mi.item_image = row.item_image,
        mi.updated_at = datetime()
    MERGE (r)-[:HAS_MENU_ITEM]->(mi)
    FOREACH (_ IN CASE WHEN row.category_name IS NULL OR row.category_name = '' THEN [] ELSE [1] END |
        MERGE (mc:MenuCategory {name: row.category_name})
        MERGE (r)-[:HAS_MENU_CATEGORY]->(mc)
        MERGE (mi)-[:IN_MENU_CATEGORY]->(mc)
    )
    """, {"rows": rows})
    print(f"Upserted {len(menu_df)} menu items")


dish_families = build_menu_dish_families(menu_items)
dish_families_by_store = dish_families.groupby("store_key")["dish_family"].apply(lambda s: sorted(set(s))).to_dict()
summary["dish_families"] = summary["store_key"].map(dish_families_by_store).apply(lambda x: x if isinstance(x, list) else [])
if dish_families.empty:
    print("No dish families found from menu")
else:
    print(f"Top dish families found:\n{dish_families.nlargest(10, 'order_count')[['store_key','dish_family','order_count','avg_price']].to_string(index=False)}")


In [ ]:
attr_rows = []
for store_key, grp in feedback_proc.groupby("store_key"):
    bucket = defaultdict(list)
    for asp_map in grp["aspect_scores"]:
        for k, v in asp_map.items():
            bucket[k].append(v)
    for aspect, vals in bucket.items():
        attr_rows.append({
            "store_key": store_key,
            "attribute_type": aspect,
            "attribute_score": round(float(np.mean(vals)), 3),
            "sample_count": len(vals)
        })
restaurant_attrs = pd.DataFrame(attr_rows)
display(restaurant_attrs.head(10))


In [ ]:
def build_text_unit_text(row: pd.Series, restaurant_name: str = "") -> str:
    aspects = ", ".join(f"{k}={v:+.2f}" for k, v in (row["aspect_scores"] or {}).items())
    return (
        f"Tên quán: {restaurant_name or row['store_name']}\n"
        f"Nguồn: {row['source']}\n"
        f"Rating người dùng: {row['rating']}\n"
        f"Sentiment tổng hợp: {row['sentiment']}\n"
        f"Aspect sentiment: {aspects}\n"
        f"Nội dung review: {row['feedback']}"
    )

name_map = summary.set_index("store_key")["name"].to_dict()
text_units = feedback_proc.copy()
text_units["text_unit_id"] = "tu_" + text_units["review_id"].astype(str)
text_units["chunk_text"] = [build_text_unit_text(r, name_map.get(r["store_key"], r["store_name"])) for _, r in text_units.iterrows()]
review_chunks = text_units.copy()  # backward-compatible alias for functions below
text_units[["store_key", "text_unit_id", "chunk_text"]].head(3)


## 4.1. Text chunking chuẩn GraphRAG (sliding window over long reviews)

In [ ]:
def chunk_review_text(
    text: str,
    chunk_size: int = 400,
    overlap: int = 80,
) -> List[str]:
    """FIX (Vấn đề 1): Proper sliding window text chunking.

    Thay vì review = 1 TextUnit, chia review dài thành nhiều chunk:
    - chunk_size ~400 ký tự ≈ 300-500 token Vietnamese
    - overlap 80 ký tự để giữ context continuity giữa các chunk
    - Review ngắn (< chunk_size): vẫn ra 1 chunk, không thay đổi behavior
    """
    text = text.strip()
    if not text:
        return []
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        if end >= len(text):
            chunks.append(text[start:])
            break
        # Ưu tiên cắt tại ranh giới câu
        boundary = max(
            text.rfind(".", start, end),
            text.rfind("!", start, end),
            text.rfind("?", start, end),
            text.rfind("\n", start, end),
        )
        if boundary > start + overlap:
            end = boundary + 1
        chunks.append(text[start:end])
        start = end - overlap
    return [c.strip() for c in chunks if c.strip()]


def build_text_units_with_chunking(
    feedback_proc_df: pd.DataFrame,
    name_map: Dict[str, str],
    chunk_size: int = 400,
    overlap: int = 80,
) -> pd.DataFrame:
    """Build TextUnit DataFrame với proper sliding window chunking.

    Mỗi review dài có thể tạo ra nhiều TextUnit chunk.
    text_unit_id = review_id + "_chunk{i}" cho multi-chunk reviews.
    """
    rows = []
    for _, r in feedback_proc_df.iterrows():
        review_text = str(r["feedback"])
        aspect_str = ", ".join(
            f"{k}={v:+.2f}" for k, v in (r["aspect_scores"] or {}).items()
        )
        header = (
            f"Tên quán: {name_map.get(r['store_key'], r['store_name'])}\n"
            f"Nguồn: {r['source']}\n"
            f"Rating người dùng: {r['rating']}\n"
            f"Sentiment tổng hợp: {r['sentiment']}\n"
            f"Aspect sentiment: {aspect_str}\n"
            f"Nội dung review: "
        )
        chunks = chunk_review_text(review_text, chunk_size=chunk_size, overlap=overlap)
        if not chunks:
            chunks = ["(empty review)"]

        for i, chunk in enumerate(chunks):
            chunk_id = f"{r['review_id']}_{i}" if len(chunks) > 1 else r["review_id"]
            rows.append({
                "text_unit_id": "tu_" + chunk_id,
                "review_id": r["review_id"],
                "store_key": r["store_key"],
                "store_name": r["store_name"],
                "rating": r["rating"],
                "rated_at": r.get("rated_at", None),
                "sentiment": r["sentiment"],
                "aspect_scores": r["aspect_scores"],
                "source": r["source"],
                "feedback": review_text,
                "chunk_text": header + chunk,
                "chunk_index": i,
                "n_chunks": len(chunks),
            })
    return pd.DataFrame(rows)


# Rebuild text_units with proper chunking (replaces flat 1-review-1-unit)
text_units = build_text_units_with_chunking(feedback_proc, name_map)
review_chunks = text_units.copy()
multi_chunk = (text_units["n_chunks"] > 1).sum()
print(f"✅ TextUnits after chunking: {len(text_units)} chunks from {len(feedback_proc)} reviews")
print(f"   Reviews producing multiple chunks: {multi_chunk}")
text_units[["store_key", "text_unit_id", "chunk_index", "n_chunks"]].head(5)

## 5. Graph schema sâu hơn

In [ ]:
GRAPH_SCHEMA_NOTE = """
GraphRAG-style schema used in this notebook

Core indexing nodes:
- (:Restaurant)        domain entity
- (:Review)            raw BeFood comment record
- (:TextUnit)          chunk/text unit; each comment is represented as one text unit
- (:Attribute)         aspect sentiment aggregate per restaurant
- (:MenuItem)          concrete menu item from BeFood
- (:DishFamily)        normalized dish/food entity derived from menu item names
- (:Community)         detected cluster over the restaurant similarity graph
- (:CommunityReport)   LLM-written summary of a community for global/contextual retrieval

Domain/context nodes:
- (:Area), (:Cuisine), (:Category), (:PriceBand), (:AtmosphereTag), (:MenuCategory)

Core relations:
- (Review)-[:HAS_TEXT_UNIT]->(TextUnit)
- (TextUnit)-[:ABOUT]->(Restaurant)
- (TextUnit)-[:MENTIONS_ASPECT]->(Attribute)
- (Restaurant)-[:HAS_ATTRIBUTE]->(Attribute)
- (Restaurant)-[:HAS_MENU_ITEM]->(MenuItem)
- (MenuItem)-[:IN_MENU_CATEGORY]->(MenuCategory)
- (Restaurant)-[:SERVES_FAMILY {menu_item_count, order_count, like_count, avg_price}]->(DishFamily)
- (Restaurant)-[:IN_COMMUNITY]->(Community)
- (Community)-[:HAS_REPORT]->(CommunityReport)
- (Restaurant)-[:SIMILAR_TO {similarity, method:'embedding_cosine'}]-(Restaurant)
"""
print(GRAPH_SCHEMA_NOTE)


In [ ]:
from neo4j import GraphDatabase

class Neo4jClient:
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.driver.verify_connectivity()

    def close(self):
        self.driver.close()

    def run(self, query: str, params: dict | None = None):
        with self.driver.session() as s:
            return [r.data() for r in s.run(query, params or {})]

    def create_schema(self):
        stmts = [
            # Drop old Enterprise-only / conflicting schema if it exists
            "DROP CONSTRAINT attr_key IF EXISTS",
            "DROP CONSTRAINT area_key IF EXISTS",

            # Community-compatible constraints
            "CREATE CONSTRAINT restaurant_key IF NOT EXISTS FOR (r:Restaurant) REQUIRE r.store_key IS UNIQUE",
            "CREATE CONSTRAINT review_key IF NOT EXISTS FOR (r:Review) REQUIRE r.review_id IS UNIQUE",
            "CREATE CONSTRAINT text_unit_key IF NOT EXISTS FOR (t:TextUnit) REQUIRE t.text_unit_id IS UNIQUE",

            # Neo4j Community does NOT support NODE KEY.
            # Use synthetic unique ids instead.
            "CREATE CONSTRAINT attr_id_key IF NOT EXISTS FOR (a:Attribute) REQUIRE a.attribute_id IS UNIQUE",
            "CREATE CONSTRAINT area_id_key IF NOT EXISTS FOR (a:Area) REQUIRE a.area_id IS UNIQUE",

            "CREATE CONSTRAINT cuisine_name IF NOT EXISTS FOR (c:Cuisine) REQUIRE c.name IS UNIQUE",
            "CREATE CONSTRAINT category_name IF NOT EXISTS FOR (c:Category) REQUIRE c.name IS UNIQUE",
            "CREATE CONSTRAINT priceband_name IF NOT EXISTS FOR (p:PriceBand) REQUIRE p.name IS UNIQUE",
            "CREATE CONSTRAINT atmos_name IF NOT EXISTS FOR (a:AtmosphereTag) REQUIRE a.name IS UNIQUE",
            "CREATE CONSTRAINT dish_family_name IF NOT EXISTS FOR (d:DishFamily) REQUIRE d.name IS UNIQUE",
                        "CREATE CONSTRAINT menu_item_key IF NOT EXISTS FOR (m:MenuItem) REQUIRE m.menu_item_id IS UNIQUE",
            "CREATE CONSTRAINT menu_category_name IF NOT EXISTS FOR (m:MenuCategory) REQUIRE m.name IS UNIQUE",
            "CREATE CONSTRAINT community_key IF NOT EXISTS FOR (c:Community) REQUIRE c.community_id IS UNIQUE",
            "CREATE CONSTRAINT community_report_key IF NOT EXISTS FOR (cr:CommunityReport) REQUIRE cr.report_id IS UNIQUE",

            "CREATE INDEX rest_rating IF NOT EXISTS FOR (r:Restaurant) ON (r.gmaps_rating)",
            "CREATE INDEX text_unit_store IF NOT EXISTS FOR (t:TextUnit) ON (t.store_key)",
            "CREATE INDEX menu_item_price IF NOT EXISTS FOR (m:MenuItem) ON (m.price)",
            "CREATE INDEX attr_type IF NOT EXISTS FOR (a:Attribute) ON (a.type)",
            "CREATE INDEX community_level IF NOT EXISTS FOR (c:Community) ON (c.level)",
        ]

        for q in stmts:
            self.run(q)


neo4j_client = Neo4jClient(
    NEO4J_URI,
    NEO4J_USER,
    NEO4J_PASSWORD,
)

neo4j_client.create_schema()
print("✅ Neo4j connected and schema is ready")

In [ ]:
# Optional schema inspection. Run manually if you need to inspect Neo4j indexes.
# with neo4j_client.driver.session() as session:
#     rows = session.run("SHOW INDEXES YIELD name, type, entityType, labelsOrTypes, properties RETURN *").data()
# rows


In [ ]:
# No destructive schema changes in the default notebook path.
# If an index/constraint must be dropped, do it explicitly in a separate maintenance cell.


In [ ]:
def upsert_restaurants_graph(client: Neo4jClient, restaurants_df: pd.DataFrame):
    def _clean_str(x: Any):
        if x is None or pd.isna(x):
            return None
        s = str(x).strip()
        return s if s else None

    def _clean_float(x: Any):
        if x is None or pd.isna(x):
            return None
        try:
            return float(x)
        except Exception:
            return None

    def _clean_int(x: Any):
        if x is None or pd.isna(x):
            return None
        try:
            return int(x)
        except Exception:
            return None

    rows = []
    for _, r in restaurants_df.iterrows():
        district = _clean_str(r.get("district"))
        city = _clean_str(r.get("city"))
        price_band = _clean_str(r.get("price_band"))

        area_id = None
        if city and district:
            area_id = f"{city}:{district}"

        rows.append({
            "store_key": str(r.get("store_key")),
            "name": _clean_str(r.get("name")) or "",
            "address": _clean_str(r.get("address")),
            "district": district,
            "city": city,
            "area_id": area_id,
            "lat": _clean_float(r.get("lat")),
            "lng": _clean_float(r.get("lng")),
            "distance_km": _clean_float(r.get("distance_km")),
            "gmaps_rating": _clean_float(r.get("gmaps_rating")),
            "foody_rating": _clean_float(r.get("foody_rating")),
            "rating": _clean_float(r.get("rating")),
            "review_count": _clean_int(r.get("review_count")),
            "price_band": price_band,
            "price_min": _clean_float(r.get("price_min")),
            "price_max": _clean_float(r.get("price_max")),
            "menu_item_count": int(_clean_int(r.get("menu_item_count")) or 0),
            "menu_price_min": _clean_float(r.get("menu_price_min")),
            "menu_price_max": _clean_float(r.get("menu_price_max")),
            "menu_price_median": _clean_float(r.get("menu_price_median")),
            "top_menu_items": r.get("top_menu_items") if isinstance(r.get("top_menu_items"), list) else [],
            "opening_hours": _clean_str(r.get("opening_hours")),
            "delivery_time": _clean_int(r.get("delivery_time")),
            "image_url": _clean_str(r.get("image_url")),
            "categories": r.get("categories"),
            "cuisines": r.get("cuisines"),
            "atmosphere": r.get("atmosphere"),
            "audiences": r.get("audiences"),
        })

    client.run("""
    UNWIND $rows AS row
    MERGE (r:Restaurant {store_key: row.store_key})
    SET r.name = row.name, r.address = row.address, r.district = row.district, r.city = row.city,
        r.lat = row.lat, r.lng = row.lng, r.user_distance_km = row.distance_km,
        r.gmaps_rating = row.gmaps_rating, r.foody_rating = row.foody_rating,
        r.rating = row.rating, r.review_count = row.review_count,
        r.price_band = row.price_band,
        r.price_min = row.price_min, r.price_max = row.price_max,
        r.menu_item_count = row.menu_item_count, r.menu_price_min = row.menu_price_min,
        r.menu_price_max = row.menu_price_max, r.menu_price_median = row.menu_price_median,
        r.top_menu_items = row.top_menu_items, r.opening_hours = row.opening_hours,
        r.delivery_time = row.delivery_time, r.image_url = row.image_url, r.audiences = row.audiences,
        r.updated_at = datetime()
    WITH r, row
    FOREACH (cat IN coalesce(row.categories, []) | MERGE (c:Category {name: cat}) MERGE (r)-[:HAS_CATEGORY]->(c))
    FOREACH (cui IN coalesce(row.cuisines, []) | MERGE (c:Cuisine {name: cui}) MERGE (r)-[:HAS_CUISINE]->(c))
    FOREACH (atm IN coalesce(row.atmosphere, []) | MERGE (a:AtmosphereTag {name: atm}) MERGE (r)-[:HAS_ATMOSPHERE]->(a))

    // Prevent NaN/empty from creating (:PriceBand {name: NaN})
    FOREACH (_ IN CASE
        WHEN row.price_band IS NULL THEN []
        WHEN trim(toString(row.price_band)) = '' THEN []
        ELSE [1]
    END |
        MERGE (p:PriceBand {name: row.price_band})
        MERGE (r)-[:HAS_PRICE_BAND]->(p)
    )

    // Prevent NaN/empty area data
    FOREACH (_ IN CASE
        WHEN row.area_id IS NULL THEN []
        WHEN row.district IS NULL OR row.city IS NULL THEN []
        WHEN trim(toString(row.district)) = '' OR trim(toString(row.city)) = '' THEN []
        ELSE [1]
    END |
        MERGE (a:Area {area_id: row.area_id})
        SET a.name = row.district, a.city = row.city
        MERGE (r)-[:IN_AREA]->(a)
    )
    """, {"rows": rows})


def upsert_attributes_graph(client: Neo4jClient, attrs_df: pd.DataFrame):
    rows = []
    for _, r in attrs_df.iterrows():
        rows.append({
            "attribute_id": f"{r['store_key']}:{r['attribute_type']}",
            "store_key": r["store_key"], "attribute_type": r["attribute_type"],
            "attribute_score": r["attribute_score"], "sample_count": r["sample_count"],
        })
    client.run("""
    UNWIND $rows AS row
    MATCH (r:Restaurant {store_key: row.store_key})
    MERGE (a:Attribute {attribute_id: row.attribute_id})
    SET a.store_key = row.store_key, a.type = row.attribute_type, a.score = row.attribute_score,
        a.sample_count = row.sample_count, a.updated_at = datetime()
    MERGE (r)-[:HAS_ATTRIBUTE]->(a)
    """, {"rows": rows})


def upsert_reviews_and_text_units_graph(client: Neo4jClient, units_df: pd.DataFrame):
    rows = []
    for _, r in units_df.iterrows():
        rows.append({
            "review_id": r["review_id"], "text_unit_id": r["text_unit_id"], "store_key": r["store_key"],
            "feedback": r["feedback"], "chunk_text": r["chunk_text"], "rating": r["rating"],
            "rated_at": str(r["rated_at"]), "sentiment": r["sentiment"],
            "aspect_scores": json.dumps(r["aspect_scores"], ensure_ascii=False), "source": r["source"],
        })
    client.run("""
    UNWIND $rows AS row
    MATCH (rest:Restaurant {store_key: row.store_key})
    MERGE (rv:Review {review_id: row.review_id})
    SET rv.feedback = row.feedback, rv.rating = row.rating, rv.rated_at = row.rated_at,
        rv.sentiment = row.sentiment, rv.aspect_scores = row.aspect_scores, rv.source = row.source
    MERGE (tu:TextUnit {text_unit_id: row.text_unit_id})
    SET tu.text = row.chunk_text, tu.store_key = row.store_key, tu.source = row.source,
        tu.review_id = row.review_id, tu.sentiment = row.sentiment, tu.rating = row.rating, tu.updated_at = datetime()
    MERGE (rv)-[:HAS_TEXT_UNIT]->(tu)
    MERGE (tu)-[:ABOUT]->(rest)
    WITH tu, row
    MATCH (att:Attribute {store_key: row.store_key})
    MERGE (tu)-[:MENTIONS_ASPECT]->(att)
    """, {"rows": rows})


upsert_restaurants_graph(neo4j_client, summary)
upsert_attributes_graph(neo4j_client, restaurant_attrs)
upsert_reviews_and_text_units_graph(neo4j_client, text_units)
upsert_menu_items_graph(neo4j_client, menu_items)
upsert_dish_families(neo4j_client, dish_families)

print("Graph upsert complete: Restaurant, Review, TextUnit, Attribute, MenuItem, DishFamily and domain nodes")


## 6. Embedding similarity edges + Leiden communities


In [ ]:
# Similarity edges and communities are built after embeddings are created in Section 7.1.


## 7. Vietnamese embeddings và Qdrant indexing


In [ ]:
import sys, torch
print(sys.executable)
print(torch.__version__)
print("cuda:", torch.version.cuda)
print("is_available:", torch.cuda.is_available())

In [ ]:
import torch 
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("Loading Vietnamese embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL)
EMBED_DIM = len(embed_model.encode("kiểm tra chiều embedding", normalize_embeddings=True))
print("✅ Embedding dim:", EMBED_DIM)

def _embed(text: str, prefix: str = "") -> List[float]:
    return embed_model.encode(prefix + text, normalize_embeddings=True).tolist()

def emb_passage(text: str) -> List[float]:
    return _embed(text, EMBED_PREFIX_PASSAGE)

def emb_query(text: str) -> List[float]:
    return _embed(text, EMBED_PREFIX_QUERY)

qdrant = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
qdrant.get_collections()  # fail fast if the service is unavailable
print("✅ Qdrant connected")

def ensure_collection(client: QdrantClient, name: str, dim: int, recreate: bool = RECREATE_QDRANT):
    existing = {c.name for c in client.get_collections().collections}
    if name in existing and recreate:
        client.delete_collection(name)
    if name not in existing or recreate:
        client.create_collection(name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
        print("Created collection:", name)
    else:
        print("Using existing collection:", name)

ensure_collection(qdrant, COLL_TEXT_UNIT, EMBED_DIM)
ensure_collection(qdrant, COLL_RESTAURANT, EMBED_DIM)


In [ ]:
%pip install pywin32

In [ ]:
def build_restaurant_summary_doc(store_key: str) -> str:
    row = summary.set_index("store_key").loc[store_key]
    attrs = restaurant_attrs[restaurant_attrs["store_key"].eq(store_key)]
    attr_text = ", ".join(f"{r.attribute_type}={r.attribute_score:+.2f}" for _, r in attrs.sort_values("attribute_type").iterrows())
    menu_text = ", ".join(row["top_menu_items"] or [])
    return "\n".join([
        f"Restaurant name: {row['name']}",
        f"Address: {row['address']}",
        f"Area: {row['district']}, {row['city']}",
        f"Distance from user: {row['distance_km']} km",
        f"Rating: {row['rating']}",
        f"Review count: {row['review_count']}",
        f"Price band: {row['price_band']}",
        f"Source price range: {row['price_min']} - {row['price_max']}",
        f"Menu price range: {row['menu_price_min']} - {row['menu_price_max']}, median={row['menu_price_median']}",
        f"Categories: {', '.join(row['categories'] or [])}",
        f"Cuisines: {', '.join(row['cuisines'] or [])}",
        f"Dish families: {', '.join(row.get('dish_families') or [])}",
        f"Top menu items: {menu_text}",
        f"Opening hours: {row['opening_hours']}",
        f"Delivery time estimate: {row['delivery_time']}",
        f"Atmosphere: {', '.join(row['atmosphere'] or [])}",
        f"Audience: {', '.join(row['audiences'] or [])}",
        f"Aggregated aspect sentiment: {attr_text}",
    ])

restaurant_docs = pd.DataFrame({
    "store_key": summary["store_key"],
    "doc_text": summary["store_key"].apply(build_restaurant_summary_doc),
})
restaurant_docs["embedding"] = [emb_passage(x) for x in tqdm(restaurant_docs["doc_text"], desc="Embed restaurants")]
text_units["embedding"] = [emb_passage(x) for x in tqdm(text_units["chunk_text"], desc="Embed text units")]

def stable_int_id(text: str) -> int:
    return int(hashlib.md5(text.encode("utf-8")).hexdigest()[:8], 16)

def index_restaurant_docs():
    points = []
    meta_by_key = summary.set_index("store_key")
    for _, row in restaurant_docs.iterrows():
        meta = meta_by_key.loc[row["store_key"]]
        points.append(PointStruct(
            id=stable_int_id("rest-" + row["store_key"]),
            vector=row["embedding"],
            payload={
                "store_key": row["store_key"], "name": meta["name"], "address": meta["address"],
                "district": meta["district"], "city": meta["city"], "rating": meta["rating"],
                "lat": meta["lat"], "lng": meta["lng"], "distance_km": meta["distance_km"],
                "price_band": meta["price_band"], "top_menu_items": meta["top_menu_items"],
                "dish_families": meta.get("dish_families"), "categories": meta.get("categories"),
                "cuisines": meta.get("cuisines"), "menu_budget_item_ratio": meta.get("menu_budget_item_ratio"),
                "menu_price_min": meta["menu_price_min"], "menu_price_max": meta["menu_price_max"],
                "menu_price_median": meta.get("menu_price_median"),
                "doc_text": row["doc_text"], "doc_type": "restaurant_summary",
            }
        ))
    qdrant.upsert(collection_name=COLL_RESTAURANT, points=points)
    print(f"Indexed restaurant docs: {len(points)}")

def index_text_units():
    points = []
    for _, row in text_units.iterrows():
        points.append(PointStruct(
            id=stable_int_id("tu-" + row["text_unit_id"]),
            vector=row["embedding"],
            payload={
                "text_unit_id": row["text_unit_id"], "review_id": row["review_id"], "store_key": row["store_key"],
                "store_name": name_map.get(row["store_key"], row["store_name"]), "rating": row["rating"],
                "sentiment": row["sentiment"], "feedback": row["feedback"], "aspect_scores": row["aspect_scores"],
                "doc_text": row["chunk_text"], "doc_type": "text_unit",
            }
        ))
    qdrant.upsert(collection_name=COLL_TEXT_UNIT, points=points)
    print(f"Indexed text units: {len(points)}")

index_restaurant_docs()
index_text_units()


## 7.1. Embedding similarity edges + Leiden communities


In [ ]:
def cosine_np(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-9))


def build_similarity_edges_from_embeddings(
    client: Neo4jClient,
    docs_df: pd.DataFrame,
    top_k: int = SIMILARITY_TOP_K,
    min_score: float = SIMILARITY_MIN_SCORE,
):
    """Build an undirected kNN similarity graph from restaurant embeddings.

    Important detail: each restaurant gets to choose its own top-k neighbors first.
    Then edges are deduplicated as undirected pairs. This avoids the old bug where
    `i > j` skipped half of the candidates before a node could select neighbors,
    which made the graph too sparse and produced ~1 community per restaurant.
    """
    keys = docs_df["store_key"].astype(str).tolist()
    mat = np.asarray(docs_df["embedding"].tolist(), dtype=np.float32)
    mat = mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-9)
    sim = mat @ mat.T

    edge_by_pair: Dict[Tuple[str, str], float] = {}
    for i, a in enumerate(keys):
        order = np.argsort(-sim[i])
        kept = 0
        for j in order:
            if i == j:
                continue
            score = float(sim[i, j])
            if score < min_score:
                continue
            b = keys[j]
            pair = tuple(sorted((a, b)))
            edge_by_pair[pair] = max(edge_by_pair.get(pair, -1.0), score)
            kept += 1
            if kept >= top_k:
                break

    edges = [
        {"a": a, "b": b, "sim": round(score, 4)}
        for (a, b), score in edge_by_pair.items()
    ]

    client.run("MATCH (:Restaurant)-[s:SIMILAR_TO]-(:Restaurant) DELETE s")
    if edges:
        client.run("""
        UNWIND $edges AS e
        MATCH (a:Restaurant {store_key: e.a})
        MATCH (b:Restaurant {store_key: e.b})
        MERGE (a)-[s:SIMILAR_TO]-(b)
        SET s.similarity = e.sim, s.method = 'embedding_cosine_knn', s.updated_at = datetime()
        """, {"edges": edges})

    degrees = defaultdict(int)
    for e in edges:
        degrees[e["a"]] += 1
        degrees[e["b"]] += 1
    isolated = len([k for k in keys if degrees[k] == 0])
    avg_degree = round((sum(degrees.values()) / max(len(keys), 1)), 2)
    print(f"Similarity graph stats: nodes={len(keys)}, edges={len(edges)}, isolated={isolated}, avg_degree={avg_degree}, min_score={min_score}, top_k={top_k}")
    return len(edges), edges


def cleanup_communities(client: Neo4jClient):
    """Remove stale communities/reports before writing a fresh Leiden partition."""
    client.run("MATCH (:Restaurant)-[rel:IN_COMMUNITY]->(:Community) DELETE rel")
    client.run("MATCH (cr:CommunityReport) DETACH DELETE cr")
    client.run("MATCH (c:Community) DETACH DELETE c")


def build_communities_igraph(
    client: Neo4jClient,
    keys: List[str],
    edges: List[dict],
    level: int = COMMUNITY_LEVEL,
) -> List[dict]:
    """Leiden community detection over the embedding kNN graph."""
    import igraph as ig

    cleanup_communities(client)

    if not edges:
        print("No similarity edges - skipping community detection")
        return []

    key_to_idx = {str(k): i for i, k in enumerate(keys)}
    g = ig.Graph(n=len(keys), directed=False)
    g.vs["store_key"] = [str(k) for k in keys]

    edge_list = []
    weights = []
    seen_pairs = set()
    for e in edges:
        a, b = str(e["a"]), str(e["b"])
        if a not in key_to_idx or b not in key_to_idx:
            continue
        pair = tuple(sorted((key_to_idx[a], key_to_idx[b])))
        if pair in seen_pairs:
            continue
        seen_pairs.add(pair)
        edge_list.append(pair)
        weights.append(float(e["sim"]))

    g.add_edges(edge_list)
    g.es["weight"] = weights

    components = g.components()
    isolated = sum(1 for comp in components if len(comp) == 1)
    print(f"Community input graph: vertices={g.vcount()}, edges={g.ecount()}, components={len(components)}, isolated_components={isolated}")

    partition = g.community_leiden(
        weights="weight",
        objective_function="modularity",
        n_iterations=10,
    )

    community_map = {}
    for community_id, members in enumerate(partition):
        for idx in members:
            community_map[str(keys[idx])] = str(community_id)

    rows = [{"store_key": k, "community_id": cid} for k, cid in community_map.items()]
    client.run("""
    UNWIND $rows AS row
    MATCH (r:Restaurant {store_key: row.store_key})
    SET r.community_id = row.community_id
    WITH r, row
    MERGE (c:Community {community_id: row.community_id})
    SET c.level = $level,
        c.algorithm = 'igraph.leiden.embedding_knn',
        c.updated_at = datetime()
    MERGE (r)-[:IN_COMMUNITY]->(c)
    """, {"rows": rows, "level": level})

    result = client.run("""
    MATCH (c:Community)<-[:IN_COMMUNITY]-(r:Restaurant)
    RETURN c.community_id AS community_id, count(r) AS size
    ORDER BY size DESC
    """)
    print(f"Leiden result: {len(partition)} communities, modularity={partition.modularity:.4f}")
    return result


n_edges, edge_list = build_similarity_edges_from_embeddings(neo4j_client, restaurant_docs)
print("Embedding-based SIMILAR_TO edges:", n_edges)

communities = build_communities_igraph(neo4j_client, restaurant_docs["store_key"].astype(str).tolist(), edge_list)
print("Communities:")
display(pd.DataFrame(communities).head(10))

def get_restaurant_subgraph(store_key: str, top_reviews: int = 3) -> Dict[str, Any]:
    q = """
    MATCH (r:Restaurant {store_key: $store_key})
    OPTIONAL MATCH (r)-[:IN_AREA]->(a:Area)
    OPTIONAL MATCH (r)-[:HAS_CUISINE]->(c:Cuisine)
    OPTIONAL MATCH (r)-[:HAS_CATEGORY]->(g:Category)
    OPTIONAL MATCH (r)-[:HAS_ATMOSPHERE]->(t:AtmosphereTag)
    OPTIONAL MATCH (r)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (r)<-[:ABOUT]-(tu:TextUnit)
    WITH r, a, collect(DISTINCT c.name) AS cuisines, collect(DISTINCT g.name) AS categories,
         collect(DISTINCT t.name) AS atmos, collect(DISTINCT {type: att.type, score: att.score}) AS attrs,
         collect(DISTINCT {text_unit_id: tu.text_unit_id, text: tu.text, sentiment: tu.sentiment, rating: tu.rating})[..$top_reviews] AS text_units
    OPTIONAL MATCH (r)-[s:SIMILAR_TO]-(nbr:Restaurant)
    OPTIONAL MATCH (r)-[:IN_COMMUNITY]->(com:Community)-[:HAS_REPORT]->(rep:CommunityReport)
    WITH r, a, cuisines, categories, atmos, attrs, text_units, rep,
         collect(DISTINCT {store_key: nbr.store_key, name: nbr.name, similarity: s.similarity})[..5] AS neighbors
    RETURN r.store_key AS store_key, r.name AS name, r.address AS address, r.gmaps_rating AS rating,
           a.name AS district, a.city AS city, cuisines, categories, atmos, attrs, text_units, neighbors,
           rep.summary AS community_report
    """
    rows = neo4j_client.run(q, {"store_key": store_key, "top_reviews": top_reviews})
    return rows[0] if rows else {}

## 8. LLM-primary intent parsing bằng Pydantic structured output


In [ ]:
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

class RestaurantIntent(BaseModel):
    query_type: Literal["search", "similar", "compare", "personalized"] = "search"
    district: Optional[str] = None
    cuisines: List[str] = Field(default_factory=list)
    categories: List[str] = Field(default_factory=list)
    dish_name: Optional[str] = None
    min_rating: Optional[float] = Field(default=None, ge=0, le=5)
    max_distance_km: Optional[float] = Field(default=None, ge=0)
    price_band: Optional[Literal["budget", "mid", "premium"]] = None
    geo_intent: Literal["nearest", "nearby", "normal"] = "normal"
    required_attributes: List[Literal["food_quality", "service", "cleanliness", "packaging", "price", "space", "speed"]] = Field(default_factory=list)
    sentiment_pref: Optional[Literal["positive", "neutral", "negative"]] = None
    top_k: int = Field(default=5, ge=1, le=20)

class CommunityReport(BaseModel):
    title: str
    summary: str
    key_strengths: List[str] = Field(default_factory=list)
    cautions: List[str] = Field(default_factory=list)
    representative_restaurants: List[str] = Field(default_factory=list)

class RecommendationAnswer(BaseModel):
    answer: str

def get_llm():
    if LLM_PROVIDER == "anthropic":
        if not ANTHROPIC_API_KEY:
            raise RuntimeError("LLM_PROVIDER=anthropic but ANTHROPIC_API_KEY is missing.")
        return ChatAnthropic(model=ANTHROPIC_MODEL, api_key=ANTHROPIC_API_KEY, temperature=0)
    if LLM_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("LLM_PROVIDER=openai but OPENAI_API_KEY is missing.")
        return ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL, temperature=0)
    raise ValueError(f"Unsupported LLM_PROVIDER={LLM_PROVIDER}. Use 'openai' or 'anthropic'.")

llm = get_llm()
intent_llm = llm.with_structured_output(RestaurantIntent)
report_llm = llm.with_structured_output(CommunityReport)
answer_llm = llm.with_structured_output(RecommendationAnswer)

KNOWN_CUISINES = sorted({x for xs in summary["cuisines"] for x in (xs or [])})
KNOWN_CATEGORIES = sorted({x for xs in summary["categories"] for x in (xs or [])})
KNOWN_DISTRICTS = sorted({str(x) for x in summary["district"].dropna().unique().tolist() if str(x).strip()})

INTENT_SYSTEM = """Bạn là bộ phân tích intent cho hệ gợi ý quán ăn GraphRAG.
Trích xuất truy vấn thành schema RestaurantIntent.
Chỉ chọn district/cuisine/category nếu người dùng thật sự nêu hoặc suy ra rõ.
Các required_attributes hợp lệ: food_quality, service, cleanliness, packaging, price, space, speed.
"""
intent_prompt = ChatPromptTemplate.from_messages([
    ("system", INTENT_SYSTEM + "\nDistrict đã biết: {districts}\nCuisine đã biết: {cuisines}\nCategory đã biết: {categories}"),
    ("human", "Query: {query}"),
])

def parse_intent(query: str) -> dict:
    parsed = (intent_prompt | intent_llm).invoke({
        "query": query,
        "districts": KNOWN_DISTRICTS,
        "cuisines": KNOWN_CUISINES,
        "categories": KNOWN_CATEGORIES,
    })
    #sửa từ đây
    # Nếu LangChain trả Pydantic object
    if hasattr(parsed, "model_dump"):
        return parsed.model_dump()

    # Nếu OpenRouter/LangChain trả dict
    if isinstance(parsed, dict):
        return parsed

    # Fallback nếu trả kiểu khác
    return {
        "query_type": "search",
        "district": None,
        "cuisines": [],
        "categories": [],
        "dish_name": None,
        "min_rating": None,
        "max_distance_km": None,
        "price_band": None,
        "geo_intent": "normal",
        "required_attributes": [],
        "sentiment_pref": None,
        "top_k": 5,
        "raw": str(parsed),
    }


## 9. Graph retrieval + neighborhood / subgraph retrieval

In [ ]:
def graph_candidate_search(
    intent: dict,
    top_k: int = 10,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[dict]:
    match_lines = ["MATCH (r:Restaurant)"]
    where = []
    params = {"top_k": int(top_k)}
    lat, lng = get_user_location(user_lat, user_lng)
    max_distance = intent.get("max_distance_km") or MAX_DISTANCE_KM
    geo_intent = intent.get("geo_intent", "normal")
    has_user_location = lat is not None and lng is not None
    params.update({"user_lat": lat, "user_lng": lng, "max_distance_km": max_distance})

    if intent.get("district"):
        match_lines.append("MATCH (r)-[:IN_AREA]->(area:Area)")
        where.append("toLower(area.name) CONTAINS toLower($district)")
        params["district"] = intent["district"]
    if intent.get("price_band"):
        match_lines.append("MATCH (r)-[:HAS_PRICE_BAND]->(pb:PriceBand)")
        where.append("pb.name = $price_band")
        params["price_band"] = intent["price_band"]
    if intent.get("cuisines"):
        match_lines.append("MATCH (r)-[:HAS_CUISINE]->(cui:Cuisine)")
        where.append("cui.name IN $cuisines")
        params["cuisines"] = intent["cuisines"]
    if intent.get("categories"):
        match_lines.append("MATCH (r)-[:HAS_CATEGORY]->(cat:Category)")
        where.append("cat.name IN $categories")
        params["categories"] = intent["categories"]
    if intent.get("dish_name"):
        dish_family = normalize_dish_family(intent["dish_name"]) or intent["dish_name"]
        match_lines.append("MATCH (r)-[:SERVES_FAMILY]->(dish:DishFamily)")
        where.append("toLower(dish.name) CONTAINS toLower($dish_name)")
        params["dish_name"] = dish_family
    if intent.get("min_rating") is not None:
        where.append("coalesce(r.rating, r.gmaps_rating, r.foody_rating, 0) >= $min_rating")
        params["min_rating"] = float(intent["min_rating"])
    if has_user_location and max_distance is not None:
        where.append("r.lat IS NOT NULL AND r.lng IS NOT NULL AND point.distance(point({latitude: $user_lat, longitude: $user_lng}), point({latitude: r.lat, longitude: r.lng})) / 1000.0 <= $max_distance_km")
    for i, attr in enumerate(intent.get("required_attributes", [])):
        alias = f"att{i}"
        match_lines.append(f"MATCH (r)-[:HAS_ATTRIBUTE]->({alias}:Attribute {{type: $attr_{i}}})")
        where.append(f"{alias}.score >= 0.15")
        params[f"attr_{i}"] = attr
    q = "\n".join(match_lines) + "\n"
    if where:
        q += "WHERE " + " AND ".join(where) + "\n"
    distance_expr = "point.distance(point({latitude: $user_lat, longitude: $user_lng}), point({latitude: r.lat, longitude: r.lng})) / 1000.0" if has_user_location else "null"
    order_clause = """
    ORDER BY CASE WHEN distance_km IS NULL THEN 1 ELSE 0 END, distance_km ASC,
             CASE WHEN coalesce(r.rating, r.gmaps_rating, r.foody_rating) IS NULL THEN 1 ELSE 0 END,
             coalesce(r.rating, r.gmaps_rating, r.foody_rating) DESC
    """ if geo_intent == "nearest" else """
    ORDER BY CASE WHEN coalesce(r.rating, r.gmaps_rating, r.foody_rating) IS NULL THEN 1 ELSE 0 END,
             coalesce(r.rating, r.gmaps_rating, r.foody_rating) DESC,
             CASE WHEN distance_km IS NULL THEN 1 ELSE 0 END, distance_km ASC
    """
    q += f"""
    OPTIONAL MATCH (r)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (r)-[:IN_AREA]->(a:Area)
    OPTIONAL MATCH (r)-[:HAS_CATEGORY]->(cat_ret:Category)
    OPTIONAL MATCH (r)-[:HAS_CUISINE]->(cui_ret:Cuisine)
    OPTIONAL MATCH (r)-[:SERVES_FAMILY]->(df_ret:DishFamily)
    OPTIONAL MATCH (r)-[:IN_COMMUNITY]->(com:Community)-[:HAS_REPORT]->(rep:CommunityReport)
    WITH r, a, rep, collect(DISTINCT {{type: att.type, score: att.score}}) AS attributes,
         collect(DISTINCT cat_ret.name) AS categories,
         collect(DISTINCT cui_ret.name) AS cuisines,
         collect(DISTINCT df_ret.name) AS dish_families,
         CASE WHEN r.lat IS NULL OR r.lng IS NULL THEN null ELSE {distance_expr} END AS distance_km
    RETURN r.store_key AS store_key, r.name AS name, r.address AS address,
           a.name AS district, a.city AS city, coalesce(r.rating, r.gmaps_rating, r.foody_rating) AS rating,
           r.lat AS lat, r.lng AS lng, distance_km,
           r.price_band AS price_band, r.top_menu_items AS top_menu_items,
           r.menu_price_min AS menu_price_min, r.menu_price_max AS menu_price_max,
           r.menu_price_median AS menu_price_median,
           categories, cuisines, dish_families, attributes, rep.summary AS community_report
    {order_clause}
    LIMIT $top_k
    """
    rows = neo4j_client.run(q, params)
    return [add_user_distance_to_record(r, lat, lng) for r in rows]


def subgraph_expand_candidates(
    seed_store_keys: List[str],
    max_neighbors: int = 6,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[dict]:
    if not seed_store_keys:
        return []
    lat, lng = get_user_location(user_lat, user_lng)
    has_user_location = lat is not None and lng is not None
    distance_expr = "point.distance(point({latitude: $user_lat, longitude: $user_lng}), point({latitude: nbr.lat, longitude: nbr.lng})) / 1000.0" if has_user_location else "null"
    q = f"""
    MATCH (seed:Restaurant)-[s:SIMILAR_TO]-(nbr:Restaurant)
    WHERE seed.store_key IN $seed_keys
    OPTIONAL MATCH (nbr)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (nbr)-[:IN_AREA]->(a:Area)
    OPTIONAL MATCH (nbr)-[:HAS_CATEGORY]->(cat_ret:Category)
    OPTIONAL MATCH (nbr)-[:HAS_CUISINE]->(cui_ret:Cuisine)
    OPTIONAL MATCH (nbr)-[:SERVES_FAMILY]->(df_ret:DishFamily)
    OPTIONAL MATCH (nbr)-[:IN_COMMUNITY]->(:Community)-[:HAS_REPORT]->(rep:CommunityReport)
    WITH nbr, s, a, rep, collect(DISTINCT {{type: att.type, score: att.score}}) AS attributes,
         collect(DISTINCT cat_ret.name) AS categories,
         collect(DISTINCT cui_ret.name) AS cuisines,
         collect(DISTINCT df_ret.name) AS dish_families,
         CASE WHEN nbr.lat IS NULL OR nbr.lng IS NULL THEN null ELSE {distance_expr} END AS distance_km
    RETURN nbr.store_key AS store_key, nbr.name AS name, nbr.address AS address,
           a.name AS district, a.city AS city, coalesce(nbr.rating, nbr.gmaps_rating, nbr.foody_rating) AS rating,
           nbr.lat AS lat, nbr.lng AS lng, distance_km,
           nbr.price_band AS price_band, nbr.top_menu_items AS top_menu_items,
           nbr.menu_price_min AS menu_price_min, nbr.menu_price_max AS menu_price_max,
           nbr.menu_price_median AS menu_price_median,
           categories, cuisines, dish_families,
           s.similarity AS sim, attributes, rep.summary AS community_report
    ORDER BY sim DESC
    LIMIT $limit
    """
    rows = neo4j_client.run(q, {"seed_keys": seed_store_keys, "limit": max_neighbors, "user_lat": lat, "user_lng": lng})
    return [add_user_distance_to_record(r, lat, lng) for r in rows]


## 10. Vector retrieval cho Restaurant summary và TextUnit


In [ ]:
def vector_search_restaurants(query: str, top_k: int = 8, user_lat: Optional[float] = None, user_lng: Optional[float] = None) -> List[dict]:
    hits = qdrant.search(collection_name=COLL_RESTAURANT, query_vector=emb_query(query), limit=top_k, with_payload=True)
    rows = []
    for h in hits:
        rec = {
            "store_key": h.payload["store_key"], "name": h.payload["name"], "address": h.payload.get("address"),
            "district": h.payload.get("district"), "city": h.payload.get("city"), "rating": h.payload.get("rating"),
            "lat": h.payload.get("lat"), "lng": h.payload.get("lng"),
            "price_band": h.payload.get("price_band"), "top_menu_items": h.payload.get("top_menu_items"),
            "dish_families": h.payload.get("dish_families"), "categories": h.payload.get("categories"),
            "cuisines": h.payload.get("cuisines"),
            "menu_price_min": h.payload.get("menu_price_min"), "menu_price_max": h.payload.get("menu_price_max"),
            "menu_price_median": h.payload.get("menu_price_median"),
            "doc_text": h.payload.get("doc_text"),
            "vec_score": round(float(h.score), 4), "source": "restaurant_vector",
        }
        rows.append(add_user_distance_to_record(rec, user_lat, user_lng))
    return rows

def vector_search_text_units(query: str, top_k: int = 16, store_keys: Optional[List[str]] = None) -> List[dict]:
    hits = qdrant.search(collection_name=COLL_TEXT_UNIT, query_vector=emb_query(query), limit=top_k, with_payload=True)
    rows = []
    allow = set(store_keys) if store_keys else None
    for h in hits:
        p = h.payload
        if allow and p.get("store_key") not in allow:
            continue
        rows.append({
            "store_key": p["store_key"], "text_unit_id": p["text_unit_id"], "review_id": p["review_id"],
            "store_name": p.get("store_name"), "rating": p.get("rating"), "sentiment": p.get("sentiment"),
            "feedback": p.get("feedback"), "aspect_scores": p.get("aspect_scores"), "doc_text": p.get("doc_text"),
            "vec_score": round(float(h.score), 4), "source": "text_unit_vector",
        })
    return rows


## 11. Fusion + rerank bằng Reciprocal Rank Fusion


In [ ]:
import math
import numpy as np

def safe_float(x, default=0.0):
    try:
        if x is None:
            return default
        x = float(x)
        if math.isnan(x) or math.isinf(x):
            return default
        return x
    except Exception:
        return default

In [ ]:
def aggregate_text_unit_evidence(text_unit_hits: List[dict]) -> Dict[str, dict]:
    bucket = defaultdict(lambda: {"evidence": [], "text_unit_vec_score_max": 0.0})
    for r in text_unit_hits:
        b = bucket[r["store_key"]]
        feedback = r.get("feedback") or r.get("doc_text")
        if feedback:
            b["evidence"].append(feedback)
        b["text_unit_vec_score_max"] = max(b["text_unit_vec_score_max"], float(r.get("vec_score") or 0.0))
    return bucket

def _rrf(rank: int, k: int = RRF_K) -> float:
    return 1.0 / (k + rank)

def infer_geo_intent(query: str, intent: Optional[dict] = None) -> str:
    intent = intent or {}
    if intent.get("geo_intent") in {"nearest", "nearby", "normal"}:
        geo = intent["geo_intent"]
    else:
        geo = "normal"
    slug = slugify_vn(query)
    if any(token in slug for token in ["gan-nhat", "closest", "nearest"]):
        return "nearest"
    if any(token in slug for token in ["quanh-day", "xung-quanh", "gan-day", "gan", "nearby", "around"]):
        return "nearby" if geo != "nearest" else geo
    return geo

def _slug_contains_any(values: Any, needle: str) -> bool:
    if values is None or not needle:
        return False
    if isinstance(values, str):
        values = [values]
    needle_slug = slugify_vn(needle)
    for v in values or []:
        v_slug = slugify_vn(v)
        if needle_slug in v_slug or v_slug in needle_slug:
            return True
    return False

def _requires_direct_evidence(query: str, intent: dict) -> bool:
    slug = slugify_vn(query)
    evidence_terms = ["ngon", "sach", "review", "danh-gia", "phuc-vu", "dong-goi", "nhanh", "chat-luong", "evidence"]
    return bool(intent.get("required_attributes") or intent.get("sentiment_pref") or any(t in slug for t in evidence_terms))

def candidate_constraint_errors(candidate: dict, intent: dict, query: str) -> List[str]:
    errors = []
    dish = intent.get("dish_name")
    if dish:
        family = normalize_dish_family(dish) or dish
        if not (_slug_contains_any(candidate.get("dish_families"), family) or _slug_contains_any(candidate.get("top_menu_items"), dish)):
            errors.append(f"dish_family_mismatch:{family}")
    price_band = intent.get("price_band")
    if price_band and candidate.get("price_band") != price_band:
        errors.append(f"price_band_mismatch:{candidate.get('price_band')}!={price_band}")
    max_distance = intent.get("max_distance_km") or MAX_DISTANCE_KM
    if max_distance is not None:
        dist = candidate.get("distance_km")
        if dist is None or float(dist) > float(max_distance):
            errors.append(f"distance_mismatch:{dist}>{max_distance}")
    if intent.get("min_rating") is not None:
        rating = candidate.get("rating")
        if rating is None or float(rating) < float(intent["min_rating"]):
            errors.append(f"rating_mismatch:{rating}<{intent['min_rating']}")
    if _requires_direct_evidence(query, intent):
        has_evidence = bool(candidate.get("evidence")) or bool(candidate.get("community_report"))
        attrs = [a for a in (candidate.get("attributes") or []) if a and a.get("score") is not None and float(a.get("score") or 0) > 0]
        if not has_evidence and not attrs:
            errors.append("missing_evidence")
    return errors

def validate_post_fusion(candidates: List[dict], intent: dict, query: str) -> List[dict]:
    valid = []
    for c in candidates:
        errors = candidate_constraint_errors(c, intent, query)
        c["constraint_errors"] = errors
        c["constraint_valid"] = not errors
        if not errors:
            valid.append(c)
    return valid

def rerank_candidates(
    query: str,
    intent: dict,
    graph_hits: List[dict],
    neighbor_hits: List[dict],
    rest_vec_hits: List[dict],
    text_unit_hits: List[dict],
) -> List[dict]:
    RATING_WEIGHT = 0.10
    geo_intent = infer_geo_intent(query, intent)
    intent["geo_intent"] = geo_intent

    by_store: Dict[str, dict] = {}

    def ensure(rec: dict):
        sid = rec["store_key"]
        if sid not in by_store:
            by_store[sid] = {
                "store_key": sid,
                "name": rec.get("name"),
                "address": rec.get("address"),
                "district": rec.get("district"),
                "city": rec.get("city"),
                "rating": rec.get("rating"),
                "lat": rec.get("lat"),
                "lng": rec.get("lng"),
                "distance_km": rec.get("distance_km"),
                "distance_score": rec.get("distance_score", 0.0),
                "price_band": rec.get("price_band"),
                "top_menu_items": rec.get("top_menu_items"),
                "dish_families": rec.get("dish_families"),
                "categories": rec.get("categories"),
                "cuisines": rec.get("cuisines"),
                "attributes": rec.get("attributes") or [],
                "menu_price_min": rec.get("menu_price_min"),
                "menu_price_max": rec.get("menu_price_max"),
                "menu_price_median": rec.get("menu_price_median"),
                "rrf_score": 0.0,
                "graph_rank_score": 0.0,
                "neighbor_score": 0.0,
                "restaurant_vec_score": 0.0,
                "text_unit_vec_score": 0.0,
                "community_report": rec.get("community_report"),
                "evidence": [],
                "source_flags": set(),
            }
        else:
            cur = by_store[sid]
            cur["community_report"] = cur.get("community_report") or rec.get("community_report")
            for key in ["name", "address", "district", "city", "lat", "lng", "distance_km", "price_band", "top_menu_items", "dish_families", "categories", "cuisines", "attributes", "menu_price_min", "menu_price_max", "menu_price_median"]:
                if (cur.get(key) in [None, [], ""]) and rec.get(key) not in [None, [], ""]:
                    cur[key] = rec.get(key)
            cur["distance_score"] = max(float(cur.get("distance_score") or 0.0), float(rec.get("distance_score") or 0.0))
        return by_store[sid]

    for rank, r in enumerate(graph_hits, start=1):
        c = ensure(r)
        c["graph_rank_score"] = max(c["graph_rank_score"], _rrf(rank))
        c["rrf_score"] += _rrf(rank)
        c["source_flags"].add("graph_filter")

    for rank, r in enumerate(neighbor_hits, start=1):
        c = ensure(r)
        c["neighbor_score"] = max(c["neighbor_score"], float(r.get("sim", 0.0)))
        c["rrf_score"] += _rrf(rank)
        c["source_flags"].add("graph_neighbor")

    for rank, r in enumerate(rest_vec_hits, start=1):
        c = ensure(r)
        c["restaurant_vec_score"] = max(c["restaurant_vec_score"], float(r.get("vec_score", 0.0)))
        c["rrf_score"] += _rrf(rank)
        c["source_flags"].add("restaurant_vector")

    summary_by_key = summary.set_index("store_key")
    text_unit_bucket = aggregate_text_unit_evidence(text_unit_hits)
    for sid, info in text_unit_bucket.items():
        meta = summary_by_key.loc[sid] if sid in summary_by_key.index else {}
        c = ensure(add_user_distance_to_record({
            "store_key": sid,
            "name": name_map.get(sid),
            "address": meta.get("address") if hasattr(meta, "get") else None,
            "district": meta.get("district") if hasattr(meta, "get") else None,
            "city": meta.get("city") if hasattr(meta, "get") else None,
            "rating": meta.get("rating") if hasattr(meta, "get") else None,
            "lat": meta.get("lat") if hasattr(meta, "get") else None,
            "lng": meta.get("lng") if hasattr(meta, "get") else None,
            "price_band": meta.get("price_band") if hasattr(meta, "get") else None,
            "top_menu_items": meta.get("top_menu_items") if hasattr(meta, "get") else None,
            "dish_families": meta.get("dish_families") if hasattr(meta, "get") else None,
            "categories": meta.get("categories") if hasattr(meta, "get") else None,
            "cuisines": meta.get("cuisines") if hasattr(meta, "get") else None,
            "menu_price_min": meta.get("menu_price_min") if hasattr(meta, "get") else None,
            "menu_price_max": meta.get("menu_price_max") if hasattr(meta, "get") else None,
            "menu_price_median": meta.get("menu_price_median") if hasattr(meta, "get") else None,
        }))
        c["text_unit_vec_score"] = max(c["text_unit_vec_score"], info["text_unit_vec_score_max"])
        c["evidence"] = info["evidence"][:3]
        c["rrf_score"] += _rrf(1) * info["text_unit_vec_score_max"]
        c["source_flags"].add("text_unit_vector")

    rrf_scores = [c["rrf_score"] for c in by_store.values()]
    max_rrf = max(max(rrf_scores) if rrf_scores else 1.0, 1e-9)
    has_distance_signal = any(c.get("distance_km") is not None for c in by_store.values())
    if not has_distance_signal:
        dist_weight = 0.0
    elif geo_intent == "nearest":
        dist_weight = max(DISTANCE_WEIGHT, 0.45)
    elif geo_intent == "nearby":
        dist_weight = max(DISTANCE_WEIGHT, 0.35)
    else:
        dist_weight = min(DISTANCE_WEIGHT, 0.08)
    retrieval_weight = max(0.0, 1.0 - RATING_WEIGHT - dist_weight)

    results = []
    for c in by_store.values():
        rating = float(c["rating"] or 0.0)
        rrf_norm = c["rrf_score"] / max_rrf
        rating_norm = rating / 5.0
        dist_norm = float(c.get("distance_score") or 0.0)
        c["geo_intent"] = geo_intent
        c["score_components"] = {
            "retrieval": round(retrieval_weight * rrf_norm, 4),
            "rating": round(RATING_WEIGHT * rating_norm, 4),
            "distance": round(dist_weight * dist_norm, 4),
            "weights": {"retrieval": retrieval_weight, "rating": RATING_WEIGHT, "distance": dist_weight},
        }
        c["final_score"] = c["score_components"]["retrieval"] + c["score_components"]["rating"] + c["score_components"]["distance"]
        c["source_flags"] = sorted(c["source_flags"])
        results.append(c)

    results = validate_post_fusion(results, intent, query)
    if geo_intent == "nearest":
        results.sort(key=lambda x: (x.get("distance_km") is None, float(x.get("distance_km") or 1e9), -float(x.get("final_score") or 0)))
    else:
        results.sort(key=lambda x: x["final_score"], reverse=True)
    return results


def hybrid_retrieve(query: str, top_k: int = 5, user_lat: Optional[float] = None, user_lng: Optional[float] = None) -> Tuple[dict, List[dict]]:
    intent = parse_intent(query)
    lat, lng = get_user_location(user_lat, user_lng)
    intent["geo_intent"] = infer_geo_intent(query, intent)
    graph_hits = graph_candidate_search(intent, top_k=max(10, top_k), user_lat=lat, user_lng=lng)
    rest_vec_hits = vector_search_restaurants(query, top_k=max(10, top_k), user_lat=lat, user_lng=lng)
    seed_keys = list(dict.fromkeys([r["store_key"] for r in graph_hits[:3]] + [r["store_key"] for r in rest_vec_hits[:3]]))
    neighbor_hits = subgraph_expand_candidates(seed_keys, max_neighbors=8, user_lat=lat, user_lng=lng)
    store_scope = list({*(r["store_key"] for r in graph_hits), *(r["store_key"] for r in rest_vec_hits), *(r["store_key"] for r in neighbor_hits)})
    text_unit_hits = vector_search_text_units(query, top_k=20, store_keys=store_scope if store_scope else None)
    ranked = rerank_candidates(query, intent, graph_hits, neighbor_hits, rest_vec_hits, text_unit_hits)
    for r in ranked:
        for key in ["restaurant_vec_score", "text_unit_vec_score", "graph_rank_score", "neighbor_score", "rrf_score", "final_score", "distance_score"]:
            r[key] = safe_float(r.get(key))
        r["rating"] = safe_float(r.get("rating"), default=None)
        r["distance_km"] = safe_float(r.get("distance_km"), default=None)
    return intent, ranked[:top_k]


## 11.1. Cross-encoder reranking (BGE-Reranker)

In [ ]:
%pip install FlagEmbedding

In [ ]:
# Cross-encoder reranking after RRF/constraint validation.
try:
    from FlagEmbedding import FlagReranker
    _RERANKER_AVAILABLE = True
except ImportError:
    _RERANKER_AVAILABLE = False
    print("FlagEmbedding not installed. Cross-encoder reranking disabled. Run: pip install FlagEmbedding")

CROSS_ENCODER_MODEL = os.getenv("CROSS_ENCODER_MODEL", "BAAI/bge-reranker-base")
_reranker = None

def get_reranker():
    global _reranker
    if not _RERANKER_AVAILABLE:
        return None
    if _reranker is None:
        print(f"Loading cross-encoder: {CROSS_ENCODER_MODEL}...")
        _reranker = FlagReranker(CROSS_ENCODER_MODEL, use_fp16=torch.cuda.is_available())
        print("Cross-encoder loaded")
    return _reranker

def build_cross_encoder_passage(c: dict) -> str:
    evidence = " | ".join((c.get("evidence") or [])[:2])
    attrs = ", ".join(f"{a.get('type')}={safe_float(a.get('score')):+.2f}" for a in (c.get("attributes") or []) if a and a.get("score") is not None)
    parts = [
        f"Name: {c.get('name')}",
        f"Address: {c.get('address')}",
        f"District: {c.get('district')}, {c.get('city')}",
        f"Rating: {c.get('rating')}",
        f"Distance_km: {c.get('distance_km')}",
        f"Price_band: {c.get('price_band')}",
        f"Menu_price: {c.get('menu_price_min')} - {c.get('menu_price_max')}, median={c.get('menu_price_median')}",
        f"Categories: {', '.join(c.get('categories') or [])}",
        f"Cuisines: {', '.join(c.get('cuisines') or [])}",
        f"Dish_families: {', '.join(c.get('dish_families') or [])}",
        f"Top_menu_items: {', '.join(c.get('top_menu_items') or [])}",
        f"Attributes: {attrs}",
        f"Community_report: {c.get('community_report') or ''}",
        f"Evidence: {evidence}",
    ]
    return "\n".join([p for p in parts if p and not p.endswith("None")])[:1200]

def _minmax_normalize(values: List[float]) -> List[float]:
    if not values:
        return []
    lo, hi = min(values), max(values)
    if abs(hi - lo) < 1e-9:
        return [0.5 for _ in values]
    return [(v - lo) / (hi - lo) for v in values]

def cross_encoder_rerank(
    query: str,
    candidates: List[dict],
    top_k: int = 5,
    ce_weight: float = 0.3,
    intent: Optional[dict] = None,
) -> List[dict]:
    intent = intent or {}
    candidates = validate_post_fusion(candidates, intent, query)
    reranker = get_reranker()
    if reranker is None or not candidates:
        return candidates[:top_k]

    passages = [build_cross_encoder_passage(c) for c in candidates]
    raw = reranker.compute_score([[query, p] for p in passages], normalize=False)
    raw_scores = [float(x) for x in (raw if isinstance(raw, list) else [raw] * len(candidates))]
    ce_scores = _minmax_normalize(raw_scores)

    for c, raw_s, ce_s in zip(candidates, raw_scores, ce_scores):
        rrf_s = float(c.get("final_score") or 0.0)
        c["ce_raw_score"] = round(raw_s, 4)
        c["ce_score"] = round(float(ce_s), 4)
        c["final_score_before_ce"] = rrf_s
        c["final_score"] = (1.0 - ce_weight) * rrf_s + ce_weight * float(ce_s)

    geo_intent = infer_geo_intent(query, intent)
    if geo_intent == "nearest":
        candidates.sort(key=lambda x: (x.get("distance_km") is None, float(x.get("distance_km") or 1e9), -float(x.get("final_score") or 0)))
    else:
        candidates.sort(key=lambda x: x["final_score"], reverse=True)
    return candidates[:top_k]

_hybrid_retrieve_rrf = hybrid_retrieve

def hybrid_retrieve(
    query: str,
    top_k: int = 5,
    use_cross_encoder: bool = True,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> Tuple[dict, List[dict]]:
    intent, ranked_rrf = _hybrid_retrieve_rrf(
        query,
        top_k=max(top_k * 2, 10),
        user_lat=user_lat,
        user_lng=user_lng,
    )
    if use_cross_encoder:
        ranked = cross_encoder_rerank(query, ranked_rrf, top_k=top_k, intent=intent)
    else:
        ranked = validate_post_fusion(ranked_rrf, intent, query)[:top_k]
    return intent, ranked

print("Cross-encoder reranker module ready (lazy-loads on first use)")


## 12. Community reports + generation/recommendation


In [ ]:
# FIX (Vấn đề 9): Community report prompt mở rộng với anti-hallucination,
# hierarchy guidance, format rõ ràng — tăng chất lượng global query.
REPORT_SYSTEM = """Bạn là chuyên gia phân tích hệ thống quán ăn sử dụng phương pháp GraphRAG.
Nhiệm vụ: Viết CommunityReport cho một cụm (community) nhà hàng được phát hiện tự động.

== YÊU CẦU BẮT BUỘC ==
1. Chỉ dùng thông tin từ context cung cấp. Không được suy diễn ngoài context.
2. title: tên ngắn gọn mô tả đặc trưng cụm (ví dụ: "Cụm quán bún bò Hoàng Mai rating cao")
3. summary: 3-5 câu mô tả đặc điểm chung, khu vực địa lý, phân khúc giá, điểm nổi bật.
4. key_strengths: 2-5 điểm mạnh cụ thể với dẫn chứng từ aspect scores hoặc review.
   Ví dụ: "food_quality trung bình +0.72 — phần lớn review khen đồ ăn ngon"
5. cautions: 1-3 điểm yếu hoặc thông tin cần lưu ý (nếu context không có, để list rỗng).
6. representative_restaurants: 2-5 tên quán tiêu biểu nhất trong cụm.

== CHẤT LƯỢNG ==
- Không viết câu chung chung như "quán có không khí tốt" nếu không có evidence.
- Không lặp lại tên quán trong summary.
- Nếu cụm nhỏ (< 3 quán): ghi rõ "Cụm nhỏ, kết quả có thể không đại diện."
- Nếu context thiếu dữ liệu: ghi rõ "Dữ liệu aspect không đầy đủ" trong cautions.
- Không bịa tên món ăn, địa điểm hoặc số liệu không có trong context.

== FORMAT ==
Trả về JSON theo Pydantic schema CommunityReport.
"""

report_prompt = ChatPromptTemplate.from_messages([
    ("system", REPORT_SYSTEM),
    ("human", "Community id: {community_id}\n\nContext (JSON list of restaurants in this community):\n{context}\n\nViết CommunityReport cho community trên."),
])

def build_community_context(community_id: str, limit: int = 30) -> str:
    rows = neo4j_client.run("""
    MATCH (c:Community {community_id: $community_id})<-[:IN_COMMUNITY]-(r:Restaurant)
    OPTIONAL MATCH (r)-[:HAS_ATTRIBUTE]->(a:Attribute)
    OPTIONAL MATCH (r)<-[:ABOUT]-(tu:TextUnit)
    WITH r,
         collect(DISTINCT {type:a.type, score:a.score}) AS attrs,
         collect(DISTINCT tu.text)[..3] AS text_units
    OPTIONAL MATCH (r)-[:SERVES_FAMILY]->(df:DishFamily)
    WITH r, attrs, text_units, collect(DISTINCT df.name) AS dish_families
    RETURN r.name AS name, r.address AS address, coalesce(r.rating, r.gmaps_rating, r.foody_rating) AS rating,
           r.district AS district, r.city AS city, r.price_band AS price_band,
           r.menu_price_median AS menu_price_median, r.top_menu_items AS top_menu_items,
           dish_families, attrs, text_units
    LIMIT $limit
    """, {"community_id": str(community_id), "limit": limit})
    return json.dumps(rows, ensure_ascii=False, indent=2)

def upsert_community_reports():
    communities_list = neo4j_client.run(
        "MATCH (c:Community)<-[:IN_COMMUNITY]-(:Restaurant) RETURN c.community_id AS community_id ORDER BY community_id"
    )
    rows = []
    for c in tqdm(communities_list, desc="Community reports"):
        cid = str(c["community_id"])
        context = build_community_context(cid)
        if not context or context == "[]":
            print(f"  ⚠️  Community {cid}: empty context, skipping")
            continue
        try:
            report = (report_prompt | report_llm).invoke({"community_id": cid, "context": context})
            if hasattr(report, "model_dump"):
                report = report.model_dump()

            rows.append({
                "community_id": cid,
                "report_id": f"community_report_{cid}",
                "title": report.get("title", ""),
                "summary": report.get("summary", ""),
                "key_strengths": report.get("key_strengths", []),
                "cautions": report.get("cautions", []),
                "representative_restaurants": report.get("representative_restaurants", []),
            })
        except Exception as e:
            print(f"  ⚠️  Community {cid} report failed: {e}")
    neo4j_client.run("""
    UNWIND $rows AS row
    MATCH (c:Community {community_id: row.community_id})
    MERGE (cr:CommunityReport {report_id: row.report_id})
    SET cr.title = row.title, cr.summary = row.summary, cr.key_strengths = row.key_strengths,
        cr.cautions = row.cautions, cr.representative_restaurants = row.representative_restaurants,
        cr.updated_at = datetime()
    MERGE (c)-[:HAS_REPORT]->(cr)
    """, {"rows": rows})
    print(f"✅ Upserted community reports: {len(rows)}")


def check_community_report_coverage() -> dict:
    rows = neo4j_client.run("""
    MATCH (r:Restaurant)
    OPTIONAL MATCH (r)-[:IN_COMMUNITY]->(c:Community)
    OPTIONAL MATCH (c)-[:HAS_REPORT]->(cr:CommunityReport)
    WITH count(DISTINCT r) AS restaurants,
         count(DISTINCT c) AS communities,
         count(DISTINCT cr) AS reports,
         count(DISTINCT CASE WHEN cr IS NOT NULL THEN r END) AS restaurants_with_report
    RETURN restaurants, communities, reports, restaurants_with_report,
           CASE WHEN restaurants = 0 THEN 0.0 ELSE toFloat(restaurants_with_report) / restaurants END AS restaurant_report_coverage
    """)
    stats = rows[0] if rows else {"restaurants": 0, "communities": 0, "reports": 0, "restaurants_with_report": 0, "restaurant_report_coverage": 0.0}
    print("CommunityReport coverage:", stats)
    if stats.get("communities", 0) == 0 or stats.get("reports", 0) == 0:
        print("WARNING: CommunityReport is empty; do not claim full GraphRAG behavior until reports exist.")
    elif float(stats.get("restaurant_report_coverage") or 0) < 0.8:
        print("WARNING: CommunityReport coverage is low; global/community context may be incomplete.")
    return stats

if RUN_COMMUNITY_REPORTS:
    upsert_community_reports()
else:
    print("Skipping community report generation. Set RUN_COMMUNITY_REPORTS=true to run it.")
community_report_stats = check_community_report_coverage()

RECOMMEND_SYSTEM = """Bạn là trợ lý gợi ý quán ăn.
QUY TẮC BẮT BUỘC:
1. Chỉ dùng các quán có trong Context đã retrieval/rerank.
2. Giữ nguyên thứ tự quán theo Context, không tự reorder.
3. Giữ nguyên tên quán như trong Context, không đổi tên, không gom thành "cụm quán".
4. Nếu một quán thiếu evidence, vẫn có thể nêu nhưng phải ghi rõ "chưa có review evidence trực tiếp".
5. Không tự bịa rating, địa chỉ, review, món ăn hoặc ưu điểm không có trong Context, nếu context thiếu, nói rõ thiếu dữ liệu, không tự bịa.
6. Trả lời bằng tiếng Việt, ngắn gọn, dễ đọc.
7. Với mỗi quán, nêu:
   - Tên quán
   - Địa chỉ
   - Rating nếu có
   - Vì sao phù hợp với query
   - Evidence nếu có

- nếu context thiếu, nói rõ thiếu dữ liệu, không tự bịa.
"""
recommend_prompt = ChatPromptTemplate.from_messages([
    ("system", RECOMMEND_SYSTEM),
    ("human", "Query: {query}\nIntent: {intent}\n\nContext:\n{context}"),
])

def is_valid_result(r):
    score = safe_float(r.get("final_score"))
    evidence = r.get("evidence") or []
    rating = r.get("rating")

    return score > 0 and (evidence or rating is not None)

def format_retrieval_context(rows: List[dict]) -> str:
    rows = [
        r for r in rows
        if (r.get("evidence") and len(r.get("evidence")) > 0) or r.get("rating") is not None
    ]
    lines = []
    for i, r in enumerate(rows, 1):
        ev = " | ".join(r.get("evidence", [])[:2]) if r.get("evidence") else ""
        ce = f" | ce_score={r['ce_score']:.4f}" if r.get("ce_score") is not None else ""
        dist = f" | distance_km={r.get('distance_km'):.2f}" if r.get("distance_km") is not None else ""
        lines.append(
            f"{i}. {r.get('name')} | rating={r.get('rating')} | district={r.get('district')}{dist} | "
            f"score={r.get('final_score'):.4f}{ce} | sources={','.join(r.get('source_flags', []))}\n"
            f"   address={r.get('address')}\n"
            f"   price_band={r.get('price_band')} | dish_families={', '.join(r.get('dish_families') or [])} | categories={', '.join(r.get('categories') or [])}\n"
            f"   community_report={r.get('community_report') or ''}\n"
            f"   evidence={ev}"
        )
    return "\n\n".join(lines)

def recommend(query: str, top_k: int = 5, user_lat: Optional[float] = None, user_lng: Optional[float] = None) -> str:
    intent, ranked = hybrid_retrieve(query, top_k=top_k, user_lat=user_lat, user_lng=user_lng)
    context = format_retrieval_context(ranked)
    answer = (recommend_prompt | answer_llm).invoke({
        "query": query,
        "intent": json.dumps(intent, ensure_ascii=False),
        "context": context,
    })
    #return answer.answer
    # Nếu trả Pydantic object
    if hasattr(answer, "answer"):
        return answer.answer

    # Nếu trả dict
    if isinstance(answer, dict):
        return answer.get("answer", str(answer))

    # Nếu trả AIMessage
    if hasattr(answer, "content"):
        return answer.content

    return str(answer)


## 13. Evaluation cho retrieval quality

In [ ]:
def recall_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    rel = set(relevant); ret = retrieved[:k]
    return len(rel.intersection(ret)) / len(rel) if rel else 0.0

def mrr_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    rel = set(relevant)
    for i, sid in enumerate(retrieved[:k], start=1):
        if sid in rel:
            return 1.0 / i
    return 0.0

def ndcg_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    rel = set(relevant)
    dcg = 0.0
    for i, sid in enumerate(retrieved[:k], start=1):
        gain = 1.0 if sid in rel else 0.0
        dcg += gain / math.log2(i + 1)
    ideal_hits = min(len(rel), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def generate_eval_dataset_from_graph(
    n_queries: int = 20,
    seed: int = 42,
) -> List[dict]:
    """FIX (Vấn đề 8): Auto-generate ground truth eval dataset bằng LLM từ graph.

    Strategy:
    1. Sample n_queries community reports hoặc restaurant clusters từ Neo4j
    2. Gọi LLM sinh câu hỏi tự nhiên dựa trên context của cluster
    3. Dùng restaurants trong cluster làm relevant_store_keys (silver labels)

    Đây là "silver standard" — không phải human labels, nhưng đủ để:
    - So sánh các variant (vector-only vs graph vs hybrid)
    - Detect regression khi thay đổi pipeline
    """
    GEN_SYSTEM = """Bạn tạo câu hỏi tìm kiếm quán ăn tự nhiên bằng tiếng Việt dựa trên thông tin về một nhóm nhà hàng.
Câu hỏi phải tự nhiên như người dùng thật hỏi (không quá kỹ thuật).
Trả về JSON: {"query": "...", "reasoning": "tại sao nhà hàng này relevant"}
Chỉ trả về JSON, không giải thích thêm."""

    rows = neo4j_client.run("""
    MATCH (c:Community)-[:HAS_REPORT]->(cr:CommunityReport)
    MATCH (c)<-[:IN_COMMUNITY]-(r:Restaurant)
    WITH c.community_id AS cid, cr.summary AS summary, cr.title AS title,
         collect(r.store_key) AS store_keys, collect(r.name) AS names, count(r) AS n
    WHERE n >= 2
    RETURN cid, summary, title, store_keys, names
    ORDER BY rand()
    LIMIT $n
    """, {"n": n_queries})

    if not rows:
        print("⚠️  No community reports found. Run upsert_community_reports() first.")
        return []

    test_cases = []
    gen_llm_raw = get_llm()
    for row in tqdm(rows, desc="Generating eval queries"):
        context = f"Community: {row['title']}\nRestaurants: {', '.join(row['names'][:5])}\nSummary: {row['summary']}"
        try:
            resp = gen_llm_raw.invoke([
                {"role": "system", "content": GEN_SYSTEM},
                {"role": "user", "content": f"Context:\n{context}\n\nSinh 1 câu hỏi tìm quán ăn phù hợp với cụm này."},
            ])
            raw = resp.content.strip()
            raw = re.sub(r"```json|```", "", raw).strip()
            parsed = json.loads(raw)
            test_cases.append({
                "query": parsed["query"],
                "relevant_store_keys": row["store_keys"],
                "community_id": row["cid"],
                "reasoning": parsed.get("reasoning", ""),
            })
        except Exception as e:
            print(f"  ⚠️  Failed to generate query for community {row['cid']}: {e}")

    print(f"✅ Generated {len(test_cases)} eval queries from graph")
    return test_cases


def evaluate_retrieval(
    test_cases: Optional[List[dict]] = None,
    k: int = 5,
    auto_generate: bool = True,
    n_auto_queries: int = 20,
) -> pd.DataFrame:
    """Evaluate retrieval pipeline.

    Args:
        test_cases: User-provided labeled queries [{query, relevant_store_keys}].
                    If None and auto_generate=True, generates from graph automatically.
        auto_generate: If True and test_cases is None, auto-generate from community reports.
        n_auto_queries: Number of queries to auto-generate.
    """
    if test_cases is None or len(test_cases) == 0:
        if auto_generate:
            print("No test_cases provided. Auto-generating from community reports...")
            test_cases = generate_eval_dataset_from_graph(n_queries=n_auto_queries)
        else:
            raise ValueError(
                "test_cases is empty and auto_generate=False. "
                "Provide labeled queries or set auto_generate=True."
            )
    if not test_cases:
        raise ValueError("Could not generate test cases. Check that community reports exist.")

    rows = []
    for tc in tqdm(test_cases, desc="Evaluating"):
        _, ranked = hybrid_retrieve(tc["query"], top_k=k)
        retrieved = [r["store_key"] for r in ranked]
        rows.append({
            "query": tc["query"],
            "relevant": tc["relevant_store_keys"],
            f"recall@{k}": recall_at_k(retrieved, tc["relevant_store_keys"], k),
            f"mrr@{k}": mrr_at_k(retrieved, tc["relevant_store_keys"], k),
            f"ndcg@{k}": ndcg_at_k(retrieved, tc["relevant_store_keys"], k),
            "retrieved": retrieved,
        })
    df = pd.DataFrame(rows)
    print("\n=== Evaluation Results ===")
    print(df[[f"recall@{k}", f"mrr@{k}", f"ndcg@{k}"]].mean().round(4).to_string())
    return df


# --- Ablation using evaluate_retrieval ---
def evaluate_ablation(test_cases: Optional[List[dict]] = None, k: int = 5) -> pd.DataFrame:
    """So sánh 4 variants: vector-only, graph-only, text-unit-only, hybrid."""
    if test_cases is None:
        test_cases = generate_eval_dataset_from_graph(n_queries=15)

    def _eval_fn(retrieve_fn):
        results = []
        for tc in test_cases:
            retrieved = retrieve_fn(tc["query"], top_k=k)
            results.append({
                "recall": recall_at_k(retrieved, tc["relevant_store_keys"], k),
                "mrr": mrr_at_k(retrieved, tc["relevant_store_keys"], k),
                "ndcg": ndcg_at_k(retrieved, tc["relevant_store_keys"], k),
            })
        return pd.DataFrame(results).mean()

    variants = {
        "vector_only": lambda q, top_k: [r["store_key"] for r in vector_search_restaurants(q, top_k=top_k)],
        "graph_only": lambda q, top_k: [r["store_key"] for r in graph_candidate_search(parse_intent(q), top_k=top_k)],
        "text_unit_only": lambda q, top_k: [r["store_key"] for r in vector_search_text_units(q, top_k=top_k)],
        "hybrid_no_ce": lambda q, top_k: retrieve_hybrid(q, top_k=top_k),
        "hybrid_with_ce": lambda q, top_k: [r["store_key"] for r in hybrid_retrieve(q, top_k=top_k, use_cross_encoder=True)[1]],
    }

    records = []
    for name, fn in variants.items():
        print(f"Evaluating: {name}...")
        scores = _eval_fn(fn)
        records.append({"variant": name, **scores.to_dict()})
    df = pd.DataFrame(records).set_index("variant")
    print("\n=== Ablation Results ===")
    print(df.round(4).to_string())
    return df

## 14. Ablation gợi ý

In [ ]:
def retrieve_vector_only(query: str, top_k: int = 5) -> List[str]:
    return [r["store_key"] for r in vector_search_restaurants(query, top_k=top_k)]

def retrieve_graph_only(query: str, top_k: int = 5) -> List[str]:
    intent = parse_intent(query)
    return [r["store_key"] for r in graph_candidate_search(intent, top_k=top_k)]

def retrieve_text_unit_only(query: str, top_k: int = 5) -> List[str]:
    return [r["store_key"] for r in vector_search_text_units(query, top_k=top_k)]

def retrieve_hybrid(query: str, top_k: int = 5) -> List[str]:
    _, ranked = hybrid_retrieve(query, top_k=top_k)
    return [r["store_key"] for r in ranked]


In [ ]:
import json
from pprint import pprint

# =========================
# 1. TEST RAW LLM RESPONSE
# =========================
print("=" * 80)
print("1) RAW LLM TEST")
print("=" * 80)

try:
    raw_resp = llm.invoke("Xin chào. Hãy trả lời đúng một câu ngắn bằng tiếng Việt.")
    
    print("TYPE:", type(raw_resp))
    print("\nRAW OBJECT:")
    print(raw_resp)
    
    print("\nCONTENT:")
    print(raw_resp.content if hasattr(raw_resp, "content") else raw_resp)

except Exception as e:
    print("❌ RAW LLM ERROR")
    print(type(e).__name__)
    print(e)


# =========================
# 2. TEST STRUCTURED COMMUNITY REPORT
# =========================
print("\n" + "=" * 80)
print("2) STRUCTURED COMMUNITY REPORT TEST")
print("=" * 80)

try:
    # Lấy 1 community thật từ Neo4j
    community_rows = neo4j_client.run("""
    MATCH (c:Community)<-[:IN_COMMUNITY]-(:Restaurant)
    RETURN DISTINCT c.community_id AS community_id
    ORDER BY community_id
    LIMIT 1
    """)

    if not community_rows:
        print("❌ Không tìm thấy Community nào trong Neo4j.")
    else:
        test_cid = str(community_rows[0]["community_id"])
        print("Testing community_id:", test_cid)

        context = build_community_context(test_cid)
        
        print("\nCONTEXT SENT TO MODEL:")
        print(context[:3000])  # in 3000 ký tự đầu cho dễ nhìn
        if len(context) > 3000:
            print("\n... [context truncated] ...")

        report = (report_prompt | report_llm).invoke({
            "community_id": test_cid,
            "context": context
        })

        print("\nTYPE:", type(report))
        print("\nRAW REPORT OBJECT:")
        pprint(report)

        print("\nNORMALIZED REPORT DICT:")
        if hasattr(report, "model_dump"):
            report_dict = report.model_dump()
        elif isinstance(report, dict):
            report_dict = report
        else:
            report_dict = {"raw": str(report)}

        pprint(report_dict)

        print("\nJSON FORMAT:")
        print(json.dumps(report_dict, ensure_ascii=False, indent=2))

except Exception as e:
    print("❌ STRUCTURED REPORT ERROR")
    print(type(e).__name__)
    print(e)


# =========================
# 3. TEST INTENT PARSING
# =========================
print("\n" + "=" * 80)
print("3) INTENT PARSING TEST")
print("=" * 80)

test_query = "gợi ý quán ăn ngon, sạch sẽ, giá hợp lý quanh Hai Bà Trưng"

try:
    intent = parse_intent(test_query)

    print("QUERY:")
    print(test_query)

    print("\nINTENT TYPE:", type(intent))
    print("\nINTENT:")
    pprint(intent)

    print("\nINTENT JSON:")
    print(json.dumps(intent, ensure_ascii=False, indent=2))

except Exception as e:
    print("❌ INTENT PARSING ERROR")
    print(type(e).__name__)
    print(e)


# =========================
# 4. TEST RETRIEVAL RESULT BEFORE FINAL LLM ANSWER
# =========================
print("\n" + "=" * 80)
print("4) RETRIEVAL RESULT BEFORE FINAL ANSWER")
print("=" * 80)

try:
    intent, ranked = hybrid_retrieve(test_query, top_k=5)

    print("INTENT FROM hybrid_retrieve:")
    pprint(intent)

    print("\nRANKED RESULTS:")
    for i, r in enumerate(ranked, 1):
        print("\n" + "-" * 80)
        print(f"RANK {i}")
        pprint(r)

    print("\nFORMATTED CONTEXT SENT TO ANSWER LLM:")
    formatted_context = format_retrieval_context(ranked)
    print(formatted_context[:5000])
    if len(formatted_context) > 5000:
        print("\n... [formatted context truncated] ...")

except Exception as e:
    print("❌ RETRIEVAL ERROR")
    print(type(e).__name__)
    print(e)


# =========================
# 5. TEST FINAL RECOMMEND ANSWER
# =========================
print("\n" + "=" * 80)
print("5) FINAL RECOMMEND ANSWER")
print("=" * 80)

try:
    final_answer = recommend(test_query, top_k=5)

    print("TYPE:", type(final_answer))
    print("\nFINAL ANSWER:")
    print(final_answer)

except Exception as e:
    print("❌ RECOMMEND ERROR")
    print(type(e).__name__)
    print(e)